# The Loupe v2: Colab runner for the Phase 1 screen (CUB-200-2011, Swin-T)

1. **Runtime → Change runtime type → T4 GPU.**
2. Optional, for the v1 probe: upload the paper's Swin-B checkpoint to Drive as `MyDrive/Loupe/checkpoints/v1_swin_b.pth`.
3. **Runtime → Run all**, then approve the Google Drive prompt.

Everything is saved under `MyDrive/Loupe/`: the CUB archive cache, one folder per run (log, best checkpoint, results), and the results tables.
If Colab disconnects, reconnect and **Run all** again. Finished runs are skipped, and the interrupted run resumes from its last completed epoch, with results identical to an uninterrupted run.

Rules are in the repo's `PROTOCOL.md`. Validation is carved from train, and this screen never opens the test set.
To check the pipeline first, tick `SMOKE_TEST` (64 images, 1 epoch, written to `output/smoke`).

**Time budget (estimate).** One Swin-T epoch takes 54 s on an RTX 4060 laptop GPU; expect about 1.5–2.5 min on a T4.
A row stops early after roughly 15–35 epochs, so each row takes about 0.5–1.5 h, and the 7 screen runs take about 4–10 h over one or two Colab sessions.

*Generated by `make_colab.py` from repo `b01243e-dirty`; code hash `066c8e014f03` (each run records the same hash).
Edit the repo and regenerate rather than editing the code embedded here.*

In [ ]:
#@title Settings
DRIVE_DIR = "/content/drive/MyDrive/Loupe"  #@param {type:"string"}
ROWS = "baseline, A, B, B2, C"  #@param {type:"string"}
RUN_D = True  #@param {type:"boolean"}
SEED = 0  #@param {type:"integer"}
PROBE_V1 = True  #@param {type:"boolean"}
V1_CKPT = "/content/drive/MyDrive/Loupe/checkpoints/v1_swin_b.pth"  #@param {type:"string"}
V1_MODEL = "swin_base_patch4_window7_224"  #@param {type:"string"}
SMOKE_TEST = False  #@param {type:"boolean"}

In [ ]:
#@title Environment: GPU, timm 1.0.30, Google Drive
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip -q install timm==1.0.30
import os
from google.colab import drive
drive.mount("/content/drive")
for d in ("cache", "checkpoints", "output", "results"):
    os.makedirs(f"{DRIVE_DIR}/{d}", exist_ok=True)
os.makedirs("/content/Loupe", exist_ok=True)
os.chdir("/content/Loupe")
import timm, torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
print("timm", timm.__version__, "| torch", torch.__version__, "|", torch.cuda.get_device_name(0))

In [ ]:
#@title Write the Loupe code, config and split file (embedded from the repo)
FILES = {
    'loupe_v2.py': '"""\nloupe_v2.py -- The Loupe v2 for timm Swin (timm >= 0.9, NHWC stages; tested on timm 1.0.30).\n\nDrop-in replacement for the v1 Loupe mask CNN. Insertion point unchanged: the gate acts on the\nStage-2 output (28x28 at 224px) at the Stage-2 -> Stage-3 boundary. Plug-and-play contract:\n\n    model = build_loupe(\'swin_tiny_patch4_window7_224.ms_in1k\', **ABLATION[\'D_both\'])\n    logits = model(images)                                          # same call as the backbone\n    loss = F.cross_entropy(logits, y) + model.reg_loss(\'l1\', lam=0.05)   # or \'coverage\'\n\nFlags (one module covers every ablation row):\n  hidden    mask-head width h; default None = C2 // 4, the v1 rule (48 on Swin-T, 64 on Swin-B)\n  mask_src  \'s2\'   v1 head at 28x28: Conv3x3(C2->h) -> ReLU -> Conv1x1(h->1) -> sigmoid\n            \'s1s2\' hi-res head at 56x56: Conv3x3(C1->h)(F1) + up2(Conv1x1(C2->h)(F2)) -> ReLU\n                   -> Conv1x1 -> sigmoid, then 2x2 avg-pool to 28x28 for gating (pooling preserves\n                   mean coverage exactly, so the regularizer value is resolution-invariant)\n            \'s1\'   hi-res head from Stage-1 features only (Stage-1 semantics are weak; optional)\n  gate      \'mult\'     g = M                      (v1)\n            \'residual\' g = alpha + (1 - alpha) M  (alpha fixed, or learned inside alpha_bounds)\n  gate_pos  \'premerge\' F2\' = F2 * g, then Stage 3 unchanged (v1 placement)\n            \'postnorm\' g, divided by its per-image spatial mean, scales each 28x28 sub-token AFTER\n                       Stage-3\'s patch-merging LayerNorm (same boundary, same 28x28 gate, zero extra\n                       parameters). A uniform gate is exactly identity, so init does not disturb the\n                       pretrained network; with the residual gate, max/min gate ratio <= 1/alpha.\n\nWhy gate_pos exists: in timm, Stage 3 begins with PatchMerging = concat 2x2 group -> LayerNorm(4C)\n-> Linear. LayerNorm is scale-invariant, so with \'premerge\' any gate that is constant within a 2x2\ngroup changes nothing (forward or backward) unless it is small enough for LN\'s eps to matter.\n`python loupe_v2.py` measures this on your installed timm.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport copy\nimport math\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\ntry:\n    from timm.models._manipulate import checkpoint_seq\nexcept ImportError:  # older timm\n    from timm.models.helpers import checkpoint_seq\n\n\ndef _nchw(x: torch.Tensor) -> torch.Tensor:\n    return x.permute(0, 3, 1, 2)  # (B,H,W,C) -> (B,C,H,W), channels_last strides, conv-friendly\n\n\nclass MaskHead(nn.Module):\n    """Single-channel spatial mask. Returns M in [0,1] with shape (B,1,Hm,Wm)."""\n\n    def __init__(self, src: str, c1: int, c2: int, hidden: int, padding_mode: str = "zeros"):\n        super().__init__()\n        self.src = src\n        c_in = c2 if src == "s2" else c1\n        self.conv3 = nn.Conv2d(c_in, hidden, 3, padding=1, padding_mode=padding_mode)\n        self.lat = nn.Conv2d(c2, hidden, 1, bias=False) if src == "s1s2" else None\n        self.out = nn.Conv2d(hidden, 1, 1)\n\n    def forward(self, f1: torch.Tensor, f2: torch.Tensor) -> torch.Tensor:\n        h = self.conv3(_nchw(f2 if self.src == "s2" else f1))\n        if self.lat is not None:  # Stage-2 semantics, projected then upsampled to Stage-1 grid\n            h = h + F.interpolate(self.lat(_nchw(f2)), size=h.shape[-2:], mode="bilinear", align_corners=False)\n        return torch.sigmoid(self.out(F.relu(h)))\n\n\ndef gated_patch_merging(pm: nn.Module, x: torch.Tensor, g: torch.Tensor) -> torch.Tensor:\n    """timm PatchMerging with a per-sub-token gate applied after its LayerNorm.\n    x: (B,H,W,C), g: (B,H,W,1). The gate is reshaped exactly like x, so the 4C layout matches."""\n    B, H, W, C = x.shape\n    assert H % 2 == 0 and W % 2 == 0, "postnorm gate needs even H and W"\n    x = x.reshape(B, H // 2, 2, W // 2, 2, C).permute(0, 1, 3, 4, 2, 5).flatten(3)\n    gg = g.reshape(B, H // 2, 2, W // 2, 2, 1).permute(0, 1, 3, 4, 2, 5)\n    x = pm.norm(x)\n    x = (x.unflatten(-1, (2, 2, C)) * gg).flatten(3)\n    return pm.reduction(x)\n\n\ndef probe_transform(m: torch.Tensor, mode: str = "none", alpha: float = 0.3) -> torch.Tensor:\n    """Inference-time interventions on a trained mask m: (B,1,H,W), H and W even (permute first if\n    your v1 mask is (B,H,W,1)). Paste into v1 right before F * M to see which mechanism v1 uses.\n      \'no_erasure\'  divide each 2x2 group by its max: under premerge this is identical to v1\n                    except where LN\'s eps mattered, so any accuracy change = erasure\'s share\n      \'no_contrast\' replace each 2x2 group by its mean: removes within-group contrast\n      \'gate_off\'    M = 1 everywhere\n      \'floor\'       alpha + (1 - alpha) M, the residual gate applied post hoc"""\n    if mode == "none":\n        return m\n    if mode == "gate_off":\n        return torch.ones_like(m)\n    if mode == "floor":\n        return alpha + (1 - alpha) * m\n    mf = m.float()\n    if mode == "no_erasure":\n        mx = F.interpolate(F.max_pool2d(mf, 2), scale_factor=2, mode="nearest")\n        out = torch.where(mx > 0, mf / mx.clamp_min(1e-30), torch.ones_like(mf))  # fp16 underflow -> uniform\n    elif mode == "no_contrast":\n        out = F.interpolate(F.avg_pool2d(mf, 2), scale_factor=2, mode="nearest")\n    else:\n        raise ValueError(mode)\n    return out.to(m.dtype)\n\n\nclass LoupeSwin(nn.Module):\n    def __init__(self, backbone: nn.Module, mask_src: str = "s2", gate: str = "mult",\n                 gate_pos: str = "premerge", alpha: float = 0.3, learn_alpha: bool = False,\n                 alpha_bounds=(0.2, 0.5), hidden: int | None = None, padding_mode: str = "zeros"):\n        super().__init__()\n        assert mask_src in ("s2", "s1", "s1s2"), mask_src\n        assert gate in ("mult", "residual"), gate\n        assert gate_pos in ("premerge", "postnorm"), gate_pos\n        self.backbone = backbone\n        self.mask_src, self.gate, self.gate_pos = mask_src, gate, gate_pos\n        c1, c2 = backbone.feature_info[0]["num_chs"], backbone.feature_info[1]["num_chs"]\n        # v1 width rule; conv3 -> (lat) -> out construction order matches v1\'s RNG use for row A\n        self.head = MaskHead(mask_src, c1, c2, hidden or c2 // 4, padding_mode)\n        self.alpha_bounds = tuple(float(a) for a in alpha_bounds)\n        self.alpha_fixed = float(alpha)\n        if gate == "residual" and learn_alpha:\n            lo, hi = self.alpha_bounds\n            assert lo < alpha < hi, "init alpha must lie strictly inside alpha_bounds"\n            p = (alpha - lo) / (hi - lo)\n            self.alpha_logit = nn.Parameter(torch.tensor(math.log(p / (1 - p))))\n        else:\n            self.register_parameter("alpha_logit", None)\n        self.probe_mode = "none"  # set to a probe_transform mode for inference-time probes\n        self._m_hi = self._m = self._g = None\n        self._check_contract()\n\n    # ---- contract -------------------------------------------------------------------------\n    def _check_contract(self):\n        bb = self.backbone\n        ok = hasattr(bb, "layers") and len(bb.layers) >= 3 and hasattr(bb.layers[2], "downsample")\n        pm = bb.layers[2].downsample if ok else None\n        assert ok and hasattr(pm, "norm") and hasattr(pm, "reduction"), \\\n            "expects timm >= 0.9 SwinTransformer (Stage 3 must start with PatchMerging)"\n        w = pm.norm.weight\n        with torch.no_grad():\n            x = torch.randn(2, 4, 4, w.numel() // 4, device=w.device, dtype=w.dtype)\n            tol = 1e-5 if w.dtype == torch.float32 else 1e-2\n            same = torch.allclose(gated_patch_merging(pm, x, torch.ones_like(x[..., :1])), pm(x), atol=tol)\n        assert same, "PatchMerging layout differs in this timm version; postnorm gate would be misaligned"\n\n    def alpha(self):\n        if self.alpha_logit is None:\n            return self.alpha_fixed\n        lo, hi = self.alpha_bounds\n        return lo + (hi - lo) * torch.sigmoid(self.alpha_logit)\n\n    # ---- forward --------------------------------------------------------------------------\n    def stage12(self, x):\n        bb = self.backbone\n        f1 = bb.layers[0](bb.patch_embed(x))  # (B, H/4, W/4, C1)   56x56 at 224\n        f2 = bb.layers[1](f1)                 # (B, H/8, W/8, C2)   28x28 at 224 (v1 insertion point)\n        return f1, f2\n\n    def from_stage2(self, f2, g):\n        """g: (B, H/8, W/8, 1). Runs Stage 3 (gated as configured), Stage 4, norm, head."""\n        bb = self.backbone\n        st3 = bb.layers[2]\n        if self.gate_pos == "premerge":\n            x = st3(f2 * g)\n        else:\n            g = g / g.mean(dim=(1, 2), keepdim=True).clamp_min(1e-6)  # unit spatial mean: uniform gate == identity\n            x = gated_patch_merging(st3.downsample, f2, g)\n            x = checkpoint_seq(st3.blocks, x) if getattr(st3, "grad_checkpointing", False) else st3.blocks(x)\n        for st in bb.layers[3:]:\n            x = st(x)\n        return bb.forward_head(bb.norm(x))\n\n    def forward(self, x):\n        f1, f2 = self.stage12(x)\n        m_hi = self.head(f1, f2)                                    # (B,1,Hm,Wm)\n        m = m_hi if m_hi.shape[-2:] == f2.shape[1:3] else F.adaptive_avg_pool2d(m_hi, f2.shape[1:3])\n        m = probe_transform(m, self.probe_mode)                     # no-op unless probing\n        a = self.alpha()\n        g = m if self.gate == "mult" else a + (1 - a) * m          # (B,1,28,28)\n        g = g.permute(0, 2, 3, 1)                                   # (B,28,28,1) NHWC\n        self._m_hi, self._m, self._g = m_hi, m, g\n        return self.from_stage2(f2, g)\n\n    # ---- loss hook ------------------------------------------------------------------------\n    def reg_loss(self, kind: str = "l1", lam: float = 0.05, tau=(0.05, 0.15),\n                 lam_cov: float = 1.0, lam_l1: float = 0.0) -> torch.Tensor:\n        """Regularizer on the mask from the latest forward, in fp32.\n        \'l1\'       lam * mean(M), per-element mean over (B,1,H,W): identical to v1\'s torch.mean(attention_map).\n        \'coverage\' lam_cov * mean_b[relu(lo - c_b)^2 + relu(c_b - hi)^2] + lam_l1 * mean(M),\n                   c_b = per-image mean of M. tau=(t, t) gives the two-sided (c_b - t)^2 target."""\n        cov = self._m_hi.float().mean(dim=(1, 2, 3))\n        if kind == "l1":\n            return lam * cov.mean()\n        if kind == "coverage":\n            lo, hi = tau\n            pen = F.relu(lo - cov).pow(2) + F.relu(cov - hi).pow(2)\n            return lam_cov * pen.mean() + lam_l1 * cov.mean()\n        raise ValueError(kind)\n\n    @torch.no_grad()\n    def gate_stats(self) -> dict:\n        """Cheap per-batch diagnostics for logging (latest forward)."""\n        m = self._m.float()\n        g = self._g.float().permute(0, 3, 1, 2)\n        gmax, gmin = F.max_pool2d(g, 2), -F.max_pool2d(-g, 2)\n        s = {"coverage": m.mean().item(),\n             "frac_m_lt_1e-3": (m < 1e-3).float().mean().item(),\n             "frac_groups_erased": (gmax < 1e-3).float().mean().item(),\n             "median_group_contrast": (gmax / gmin.clamp_min(1e-6)).median().item()}\n        if self.gate == "residual":\n            s["alpha"] = float(self.alpha())\n        return s\n\n    def mask_for_eval(self, size):\n        """Mask at native resolution, upsampled to image size, for part-IoU evaluation (no training use)."""\n        return F.interpolate(self._m_hi.float(), size=size, mode="bilinear", align_corners=False)\n\n\n# ---- builders / configs -----------------------------------------------------------------------\nABLATION = {\n    "A_v1":       dict(mask_src="s2",   gate="mult",     gate_pos="premerge"),\n    "B_residual": dict(mask_src="s2",   gate="residual", gate_pos="premerge", alpha=0.3),\n    "C_hires":    dict(mask_src="s1s2", gate="mult",     gate_pos="premerge"),\n    "D_both":     dict(mask_src="s1s2", gate="residual", gate_pos="premerge", alpha=0.3),\n}\nABLATION_POSTNORM = {  # LN-aware placement; run per the probe / screen plan\n    "A2_v1_postnorm":       dict(ABLATION["A_v1"], gate_pos="postnorm"),\n    "B2_residual_postnorm": dict(ABLATION["B_residual"], gate_pos="postnorm"),\n    "D2_both_postnorm":     dict(ABLATION["D_both"], gate_pos="postnorm"),\n}\n\n\ndef build_loupe(model_name: str = "swin_tiny_patch4_window7_224", num_classes: int = 200,\n                pretrained: bool = True, **loupe_kwargs) -> LoupeSwin:\n    import timm\n    # create -> reset_classifier -> mask head: same RNG order and default nn.Linear init as v1\'s\n    # SwinWithLoupe (create_model(..., num_classes=200) would trunc_normal-init the head instead)\n    bb = timm.create_model(model_name, pretrained=pretrained)\n    bb.reset_classifier(num_classes)\n    return LoupeSwin(bb, **loupe_kwargs)\n\n\ndef load_v1_state_dict(model: LoupeSwin, sd: dict):\n    """Load a v1 SwinWithLoupe checkpoint into a row-A LoupeSwin (mask_src=\'s2\', gate=\'mult\')."""\n    ren = {"loupe.attention_net.0.": "head.conv3.", "loupe.attention_net.2.": "head.out.", "head.": "backbone.head.fc."}\n    out = {}\n    for k, v in sd.items():\n        new = next((n + k[len(o):] for o, n in ren.items() if k.startswith(o)), "backbone." + k)\n        out[new] = v\n    return model.load_state_dict(out)\n\n\ndef param_groups(model: nn.Module, lr: float, weight_decay: float, alpha_lr: float = 1e-3):\n    """v1 used one lr for everything. Lion steps are lr*sign(.), so at lr=1e-5 a learned alpha logit\n    moves at most ~0.2 in a full run (alpha barely changes). Give alpha its own lr if you learn it."""\n    alpha = [p for n, p in model.named_parameters() if n.endswith("alpha_logit")]\n    rest = [p for n, p in model.named_parameters() if not n.endswith("alpha_logit")]\n    groups = [{"params": rest, "lr": lr, "weight_decay": weight_decay}]\n    if alpha:\n        groups.append({"params": alpha, "lr": alpha_lr, "weight_decay": 0.0})\n    return groups\n\n\n# ---- self-test ---------------------------------------------------------------------------------\n@torch.no_grad()\ndef ln_check(model: LoupeSwin, x: torch.Tensor):\n    """How much do the logits move when the Stage-2 map is gated, by placement?"""\n    model.eval()\n    _, f2 = model.stage12(x)\n    B, H, W, _ = f2.shape\n    ones = f2.new_ones(B, H, W, 1)\n    pos0 = model.gate_pos\n    rows = []\n    gens = {f"uniform {c:g}": (lambda c=c: ones * c) for c in (0.3, 0.05, 1e-2, 1e-3, 1e-4)}\n    gen = torch.Generator(device=f2.device).manual_seed(0)\n    rand = lambda *shape: torch.rand(*shape, generator=gen, device=f2.device, dtype=f2.dtype)\n    blk = rand(B, 1, H // 2, W // 2) * 0.7 + 0.3                # constant within each 2x2 group, in [0.3, 1]\n    gens["2x2-constant in [0.3,1]"] = lambda: F.interpolate(blk, scale_factor=2, mode="nearest").permute(0, 2, 3, 1)\n    per_tok = rand(B, H, W, 1) * 0.7 + 0.3                       # varies inside groups too\n    gens["per-token in [0.3,1]"] = lambda: per_tok\n    for pos in ("premerge", "postnorm"):\n        model.gate_pos = pos\n        ref = model.from_stage2(f2, ones)\n        for name, gen in gens.items():\n            z = model.from_stage2(f2, gen())\n            rel = ((z - ref).norm() / ref.norm()).item()\n            agree = (z.argmax(1) == ref.argmax(1)).float().mean().item()\n            rows.append((pos, name, rel, agree))\n    model.gate_pos = pos0\n    return rows\n\n\ndef selftest(model_name: str, pretrained: bool, hidden: int, images: str = ""):\n    import glob\n    import timm\n    torch.manual_seed(0)\n    # with real images keep the pretrained classifier so top-1 agreement is meaningful\n    bb = timm.create_model(model_name, pretrained=pretrained, **({} if images else {"num_classes": 200})).eval()\n    if images:  # real photos + pretrained weights give realistic feature scales for the LN check\n        from PIL import Image\n        cfg = timm.data.resolve_data_config({}, model=bb)\n        cfg.update(crop_pct=224 / 256)  # resize 256 -> center crop 224, as in v1 testing\n        tf = timm.data.create_transform(**cfg)\n        x = torch.stack([tf(Image.open(p).convert("RGB")) for p in sorted(glob.glob(images))])\n    else:\n        x = torch.randn(2, 3, 224, 224)\n    y = torch.zeros(x.shape[0], dtype=torch.long)\n    n_bb = sum(p.numel() for p in bb.parameters())\n    with torch.no_grad():\n        ref = bb(x)\n    print(f"timm {timm.__version__} | {model_name} | pretrained={pretrained} | backbone {n_bb/1e6:.2f}M | hidden={hidden}")\n\n    for name, cfg in {**ABLATION, **ABLATION_POSTNORM}.items():\n        torch.manual_seed(0)\n        model = LoupeSwin(bb, hidden=hidden, **cfg)  # wraps the same backbone; nothing in it is modified\n        n_add = sum(p.numel() for n, p in model.named_parameters() if not n.startswith("backbone."))\n        model.eval()\n        with torch.no_grad():\n            model.probe_mode = "gate_off"\n            same = torch.allclose(model(x), ref, atol=1e-4)\n            model.probe_mode = "none"\n            z0 = model(x)\n        init_shift = ((z0 - ref).norm() / ref.norm()).item()\n        model.train()\n        bb.zero_grad(set_to_none=True)\n        loss = F.cross_entropy(model(x), y) + model.reg_loss("coverage") + model.reg_loss("l1")\n        loss.backward()\n        grads_ok = all(p.grad is not None and torch.isfinite(p.grad).all() for p in model.head.parameters())\n        print(f"{name:21s} +{n_add:6d} ({100*n_add/n_bb:.3f}%) | gate_off==backbone {same} | grads {grads_ok} | "\n              f"init logit shift {init_shift:.3f} | M_hi {tuple(model._m_hi.shape[1:])} -> M {tuple(model._m.shape[1:])}")\n    bb.zero_grad(set_to_none=True)\n    with torch.no_grad():\n        f1, f2 = model.stage12(x)\n    print(f"shapes: x {tuple(x.shape)} -> F1 {tuple(f1.shape)} -> F2 {tuple(f2.shape)} (gated) -> logits {tuple(ref.shape)}")\n\n    model = LoupeSwin(bb, hidden=hidden, learn_alpha=True, **ABLATION["D_both"]).train()\n    (F.cross_entropy(model(x), y) + model.reg_loss("coverage")).backward()\n    print(f"learned alpha={float(model.alpha().detach()):.3f}, grad={model.alpha_logit.grad.item():+.2e}")\n    bb.zero_grad(set_to_none=True)\n\n    pm = copy.deepcopy(bb.layers[2].downsample)\n    pm.norm = nn.Identity()  # without LN, postnorm gating must equal premerge gating exactly\n    f = torch.randn(2, 28, 28, bb.feature_info[1]["num_chs"])\n    g = torch.rand(2, 28, 28, 1)\n    print("postnorm layout == premerge when LN removed:", torch.allclose(gated_patch_merging(pm, f, g), pm(f * g), atol=1e-5))\n\n    print(f"Stage-2 output per-element std: {f2.std().item():.3f}")\n    print("LN check: relative logit change vs ungated, top-1 agreement")\n    for pos, name, rel, agree in ln_check(LoupeSwin(bb, hidden=hidden), x):\n        print(f"  {pos:9s} {name:26s} rel={rel:.2e} top1_agree={agree:.2f}")\n\n\nif __name__ == "__main__":\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--model", default="swin_tiny_patch4_window7_224")\n    ap.add_argument("--pretrained", action="store_true")\n    ap.add_argument("--hidden", type=int, default=None, help="default C2//4 (v1 rule)")\n    ap.add_argument("--images", default="", help="optional glob of real images, e.g. \'cub_val/*.jpg\'")\n    a = ap.parse_args()\n    selftest(a.model, a.pretrained, a.hidden, a.images)\n',
    'model.py': "import torch.nn as nn\nimport timm\nfrom loupe_v2 import probe_transform\n\nclass LoupeModule(nn.Module):\n    def __init__(self, in_channels, reduction_factor=4):\n        super().__init__()\n        self.attention_net = nn.Sequential(\n            nn.Conv2d(in_channels, in_channels // reduction_factor, kernel_size=3, padding=1),\n            nn.ReLU(),\n            nn.Conv2d(in_channels // reduction_factor, 1, kernel_size=1),\n            nn.Sigmoid()\n        )\n\n    def forward(self, x):\n        return self.attention_net(x)\n\nclass SwinWithLoupe(nn.Module):\n    def __init__(self, model_name='swin_base_patch4_window7_224', num_classes=200, pretrained=True):\n        super().__init__()\n\n        backbone = timm.create_model(model_name, pretrained=pretrained)\n\n        self.patch_embed = backbone.patch_embed\n        self.layers = backbone.layers\n        self.norm = backbone.norm\n        self.head = nn.Linear(backbone.head.in_features, num_classes)\n\n        loupe_in_channels = self.layers[1].blocks[-1].norm1.normalized_shape[0]\n        self.loupe = LoupeModule(in_channels=loupe_in_channels)\n\n    def forward(self, x):\n        x = self.patch_embed(x)\n        x = self.layers[0](x)\n        x = self.layers[1](x)\n\n        B, H, W, C = x.shape\n        features_to_inspect = x.permute(0, 3, 1, 2)\n\n        attention_map = self.loupe(features_to_inspect)\n        attention_map = probe_transform(attention_map, getattr(self, 'probe_mode', 'none'))  # inference-time probes only\n        refined_features = features_to_inspect * attention_map\n\n        x = refined_features.permute(0, 2, 3, 1)\n        x = self.layers[2](x)\n        x = self.layers[3](x)\n\n        x = x.flatten(1, 2)\n        x = self.norm(x)\n        x = x.mean(dim=1)\n\n        logits = self.head(x)\n\n        return logits, attention_map",
    'train_v2.py': '"""Harness for every Loupe v2 experiment (recipe frozen in PROTOCOL.md). Same recipe as v1 train.py\n(Lion, cosine per epoch, RandAugment, early stopping on val loss incl. the reg term), but validation is\ncarved from train, the checkpoint is chosen on validation accuracy, and test is evaluated at most once\nper final checkpoint (only when the config sets eval_test). Runs resume from <out>/<run>/last.pth, so a\nColab disconnect costs at most one epoch.\n\n    python train_v2.py --config configs/screen_cub_swin_t.json --row A\n    python train_v2.py --config configs/screen_cub_swin_t.json --row D --gate_pos postnorm\n    python train_v2.py --config configs/screen_cub_swin_t.json --summarize results/screen_cub_swin_t\n    python train_v2.py --config configs/screen_cub_swin_t.json --make_split   # once per dataset, then commit\n"""\nimport argparse\nimport csv\nimport glob\nimport hashlib\nimport json\nimport os\nimport platform\nimport random\nimport time\n\nimport numpy as np\nimport timm\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.optim.lr_scheduler import CosineAnnealingLR\nfrom torch.utils.data import DataLoader, Subset\nfrom torchvision.datasets import ImageFolder\nfrom timm.data import create_transform\nfrom timm.optim import create_optimizer_v2\nfrom tqdm import tqdm\n\nfrom loupe_v2 import build_loupe\n\nDEVICE = "cuda" if torch.cuda.is_available() else "cpu"\nHERE = os.path.dirname(os.path.abspath(__file__))\n\n\nclass Plain(nn.Module):\n    """No-Loupe baseline with the LoupeSwin training interface: same build, harness and recipe."""\n\n    def __init__(self, backbone):\n        super().__init__()\n        self.backbone = backbone\n\n    def forward(self, x):\n        return self.backbone(x)\n\n    def reg_loss(self, **kw):\n        return 0.0\n\n    def gate_stats(self):\n        return {}\n\n\ndef build_model(cfg, row, pretrained=True):\n    row = dict(row)\n    if row.pop("loupe", True):\n        return build_loupe(cfg["model"], cfg["num_classes"], pretrained=pretrained, **row)\n    bb = timm.create_model(cfg["model"], pretrained=pretrained)  # same RNG order as build_loupe\n    bb.reset_classifier(cfg["num_classes"])\n    return Plain(bb)\n\n\n# ---- data -------------------------------------------------------------------------------------\ndef make_split(cfg):\n    """Stratified per-class val split of the official train set. Written once per dataset, then committed."""\n    path, root = cfg["split_file"], os.path.join(cfg["data_dir"], "train")\n    assert not os.path.exists(path), f"{path} exists; splits are frozen (PROTOCOL.md)"\n    ds = ImageFolder(root)\n    by_cls = {}\n    for i, (_, y) in enumerate(ds.samples):\n        by_cls.setdefault(y, []).append(i)\n    rng = random.Random(cfg["split_seed"])\n    val = sorted(i for idx in by_cls.values() for i in rng.sample(idx, max(1, round(cfg["val_frac"] * len(idx)))))\n    split = {"frac": cfg["val_frac"], "seed": cfg["split_seed"], "n_train_total": len(ds.samples),\n             "n_classes": len(ds.classes), "val": [os.path.relpath(ds.samples[i][0], root) for i in val]}\n    os.makedirs(os.path.dirname(path), exist_ok=True)\n    with open(path, "w") as f:\n        json.dump(split, f, indent=0)\n    return split\n\n\ndef load_split(cfg, samples, root):\n    """Boolean val mask over `samples`; fails loudly if the data on disk does not match the split file."""\n    with open(cfg["split_file"]) as f:\n        split = json.load(f)\n    rel = [os.path.relpath(p, root) for p, _ in samples]\n    assert len(rel) == split["n_train_total"], f"{root} has {len(rel)} images, split expects {split[\'n_train_total\']}"\n    val = set(split["val"])\n    is_val = [r in val for r in rel]\n    assert sum(is_val) == len(val), "split file lists images that are not in the train folder"\n    return is_val\n\n\ndef make_loaders(cfg, g, limit=None):\n    root = os.path.join(cfg["data_dir"], "train")\n    train_tf = create_transform(input_size=cfg["img_size"], is_training=True, auto_augment=cfg["auto_augment"])\n    eval_tf = create_transform(input_size=cfg["img_size"], is_training=False)\n    train_full, val_full = ImageFolder(root, transform=train_tf), ImageFolder(root, transform=eval_tf)\n    is_val = load_split(cfg, train_full.samples, root)\n    train_ds = Subset(train_full, [i for i, v in enumerate(is_val) if not v][:limit])\n    val_ds = Subset(val_full, [i for i, v in enumerate(is_val) if v][:limit])\n    test_ds = ImageFolder(os.path.join(cfg["data_dir"], "test"), transform=eval_tf)\n    if limit:\n        test_ds = Subset(test_ds, range(limit))\n    kw = dict(batch_size=cfg["batch_size"], num_workers=cfg["num_workers"], pin_memory=True)\n    return (DataLoader(train_ds, shuffle=True, generator=g, **kw),\n            DataLoader(val_ds, shuffle=False, **kw), DataLoader(test_ds, shuffle=False, **kw))\n\n\n# ---- helpers ----------------------------------------------------------------------------------\n@torch.no_grad()\ndef evaluate(model, loader, reg, desc):\n    model.eval()\n    loss, correct, n, stats = 0.0, 0, 0, {}\n    for images, labels in tqdm(loader, desc=desc, leave=False):\n        images, labels = images.to(DEVICE), labels.to(DEVICE)\n        logits = model(images)\n        loss += (F.cross_entropy(logits, labels) + model.reg_loss(**reg)).item()\n        correct += (logits.argmax(1) == labels).sum().item()\n        n += labels.size(0)\n        for k, v in model.gate_stats().items():\n            stats[k] = stats.get(k, 0.0) + v / len(loader)\n    return loss / len(loader), correct / n, stats\n\n\ndef test_metrics(model, loader, reg):\n    loss, acc, stats = evaluate(model, loader, reg, "test (once)")\n    return {"test_acc": acc, "test_loss": loss, **{f"test_{k}": v for k, v in stats.items()}}\n\n\ndef run_meta(model):\n    """What every run records: exact pretrained tag, versions, GPU, parameter overhead, code hash."""\n    n_bb = sum(p.numel() for p in model.backbone.parameters())\n    n_add = sum(p.numel() for p in model.parameters()) - n_bb\n    code = hashlib.sha256()\n    for f in ("loupe_v2.py", "train_v2.py"):\n        with open(os.path.join(HERE, f), "rb") as fh:\n            code.update(fh.read())\n    pc = model.backbone.pretrained_cfg\n    return {"pretrained_tag": f"{pc[\'architecture\']}.{pc[\'tag\']}", "hf_hub_id": pc.get("hf_hub_id"),\n            "timm": timm.__version__, "torch": torch.__version__, "python": platform.python_version(),\n            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",\n            "backbone_params": n_bb, "added_params": n_add, "overhead_pct": 100 * n_add / n_bb,\n            "code_sha": code.hexdigest()[:12]}\n\n\ndef save_atomic(obj, path):  # a disconnect mid-write must not corrupt the checkpoint on Drive\n    torch.save(obj, path + ".tmp")\n    os.replace(path + ".tmp", path)\n\n\ndef rng_state(g):\n    return {"python": random.getstate(), "numpy": np.random.get_state(), "torch": torch.get_rng_state(),\n            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None, "loader": g.get_state()}\n\n\ndef set_rng_state(s, g):\n    random.setstate(s["python"])\n    np.random.set_state(s["numpy"])\n    torch.set_rng_state(s["torch"])\n    if s["cuda"] is not None and torch.cuda.is_available():\n        torch.cuda.set_rng_state_all(s["cuda"])\n    g.set_state(s["loader"])\n\n\ndef run_name(row_name, gate_pos, seed):\n    return f"{row_name}{\'_\' + gate_pos if gate_pos else \'\'}_seed{seed}"\n\n\ndef load_result(out, row_name, seed, gate_pos=None):\n    with open(os.path.join(out, run_name(row_name, gate_pos, seed), "results.json")) as f:\n        return json.load(f)\n\n\n# ---- training ---------------------------------------------------------------------------------\ndef train(config, row_name, gate_pos=None, seed=None, out=None, limit=None, epochs=None):\n    """Train one row and return its results. Finished runs are skipped; interrupted runs resume."""\n    with open(config) as f:\n        cfg = json.load(f)\n    cfg.update({k: v for k, v in (("seed", seed), ("epochs", epochs)) if v is not None})\n    row = dict(cfg["rows"][row_name], **({"gate_pos": gate_pos} if gate_pos else {}))\n    assert not row.get("loupe", True) or row.get("gate_pos"), \\\n        f"row {row_name}: pass gate_pos (the better of B / B2 on validation accuracy)"\n    assert timm.__version__ == cfg["timm_version"], f"timm {timm.__version__} installed, config pins {cfg[\'timm_version\']}"\n    run = run_name(row_name, gate_pos, cfg["seed"])\n    out_dir = os.path.join(out or os.path.join("output", os.path.splitext(os.path.basename(config))[0]), run)\n    os.makedirs(out_dir, exist_ok=True)\n    res_path, best_path, last_path, log_path = (os.path.join(out_dir, f)\n                                                for f in ("results.json", "best.pth", "last.pth", "log.jsonl"))\n    reg, sig = cfg["reg"], json.dumps({**cfg, "rows": None, "row": row}, sort_keys=True)\n\n    if os.path.exists(res_path):  # finished; open test now only if the config asks and it was never opened\n        with open(res_path) as f:\n            res = json.load(f)\n        if cfg["eval_test"] and "test_acc" not in res:\n            t0 = time.time()\n            model = build_model(cfg, row, pretrained=False).to(DEVICE)\n            model.load_state_dict(torch.load(best_path, map_location=DEVICE))\n            res.update(test_metrics(model, make_loaders(cfg, torch.Generator(), limit)[2], reg))\n            res["gpu_hours"] += (time.time() - t0) / 3600\n            with open(res_path, "w") as f:\n                json.dump(res, f, indent=2)\n        if os.path.exists(last_path):\n            os.remove(last_path)\n        print(f"[{run}] done: best val acc {100 * res[\'best_val_acc\']:.2f}% at epoch {res[\'best_epoch\']}"\n              + (f", test {100 * res[\'test_acc\']:.2f}%" if "test_acc" in res else ""))\n        return res\n\n    random.seed(cfg["seed"]); np.random.seed(cfg["seed"]); torch.manual_seed(cfg["seed"])\n    g = torch.Generator().manual_seed(cfg["seed"])\n    train_loader, val_loader, test_loader = make_loaders(cfg, g, limit)\n    torch.manual_seed(cfg["seed"])  # init independent of how much RNG the data setup consumed\n    model = build_model(cfg, row).to(DEVICE)\n    meta = run_meta(model)\n    optimizer = create_optimizer_v2(model, opt=cfg["opt"], lr=cfg["lr"], weight_decay=cfg["weight_decay"])\n    scheduler = CosineAnnealingLR(optimizer, T_max=cfg["epochs"], eta_min=cfg["eta_min"])\n    st = {"epoch": 0, "best_acc": -1.0, "best_epoch": -1, "best_val_loss": float("inf"), "patience": 0, "elapsed": 0.0}\n\n    if os.path.exists(last_path):\n        ck = torch.load(last_path, map_location="cpu", weights_only=False)\n        assert ck["sig"] == sig, f"config changed since {run} started; delete {out_dir} to restart it"\n        model.load_state_dict(ck["model"])\n        optimizer.load_state_dict(ck["optimizer"])\n        scheduler.load_state_dict(ck["scheduler"])\n        set_rng_state(ck["rng"], g)\n        st = ck["state"]\n        if st["best_epoch"] == st["epoch"]:  # disconnect may have hit between last.pth and best.pth\n            save_atomic(model.state_dict(), best_path)\n        with open(log_path) as f:  # drop log lines from the epoch that was cut off\n            keep = [l for l in f if json.loads(l)["epoch"] <= st["epoch"]]\n        with open(log_path, "w") as f:\n            f.writelines(keep)\n        print(f"[{run}] resuming after epoch {st[\'epoch\']}")\n    else:\n        with open(os.path.join(out_dir, "config.json"), "w") as f:\n            json.dump({**cfg, "rows": None, "row_name": row_name, "row": row, "meta": meta}, f, indent=2)\n        open(log_path, "w").close()\n        print(f"[{run}] {meta[\'pretrained_tag\']} | +{meta[\'added_params\']:,} params ({meta[\'overhead_pct\']:.3f}%) | "\n              f"{len(train_loader.dataset)} train / {len(val_loader.dataset)} val | {meta[\'gpu\']}")\n\n    accum, n_steps, t0 = cfg["accum_steps"], len(train_loader), time.time()\n    for epoch in range(st["epoch"], cfg["epochs"]):\n        if st["patience"] >= cfg["patience"]:\n            break\n        t_ep = time.time()\n        model.train()\n        train_loss = 0.0\n        optimizer.zero_grad()\n        for step, (images, labels) in enumerate(tqdm(train_loader, desc=f"{run} ep{epoch + 1} train", leave=False)):\n            images, labels = images.to(DEVICE), labels.to(DEVICE)\n            loss = F.cross_entropy(model(images), labels) + model.reg_loss(**reg)\n            group = min(accum, n_steps - step // accum * accum)  # short final group at epoch end\n            (loss / group).backward()\n            if (step + 1) % accum == 0 or step + 1 == n_steps:\n                optimizer.step()\n                optimizer.zero_grad()\n            train_loss += loss.item()\n\n        val_loss, val_acc, stats = evaluate(model, val_loader, reg, f"{run} ep{epoch + 1} val")\n        rec = {"epoch": epoch + 1, "lr": scheduler.get_last_lr()[0], "train_loss": train_loss / n_steps,\n               "val_loss": val_loss, "val_acc": val_acc, **{f"val_{k}": v for k, v in stats.items()}}\n        scheduler.step()\n        new_best = val_acc > st["best_acc"]\n        if new_best:\n            st["best_acc"], st["best_epoch"] = val_acc, epoch + 1\n        if st["best_val_loss"] - val_loss > cfg["min_delta"]:\n            st["best_val_loss"], st["patience"] = val_loss, 0\n        else:\n            st["patience"] += 1\n        st["epoch"] = epoch + 1\n        st["elapsed"] += time.time() - t0\n        t0 = time.time()\n        rec["epoch_seconds"] = round(t0 - t_ep, 1)\n        with open(log_path, "a") as f:\n            f.write(json.dumps(rec) + "\\n")\n        save_atomic({"model": model.state_dict(), "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),\n                     "rng": rng_state(g), "state": st, "sig": sig}, last_path)\n        if new_best:\n            save_atomic(model.state_dict(), best_path)\n        gate = "".join(f" | {k} {stats[k]:.3g}" for k in ("coverage", "frac_groups_erased", "median_group_contrast") if k in stats)\n        print(f"[{run}] ep {epoch + 1:2d} | train {rec[\'train_loss\']:.3f} | val loss {val_loss:.3f} acc {100 * val_acc:.2f}%"\n              f"{\' *\' if new_best else \'\'}{gate} | patience {st[\'patience\']}/{cfg[\'patience\']}")\n\n    model.load_state_dict(torch.load(best_path, map_location=DEVICE))\n    with open(log_path) as f:\n        best_rec = next(r for r in map(json.loads, f) if r["epoch"] == st["best_epoch"])\n    res = {"run": run, "row": row_name, "gate_pos": row.get("gate_pos"), "seed": cfg["seed"], **meta,\n           "epochs_run": st["epoch"], "early_stopped": st["patience"] >= cfg["patience"], "best_epoch": st["best_epoch"],\n           "best_val_acc": st["best_acc"], **{k: v for k, v in best_rec.items() if k.startswith("val_") and k != "val_acc"}}\n    if cfg["eval_test"]:\n        res.update(test_metrics(model, test_loader, reg))\n    res["gpu_hours"] = (st["elapsed"] + time.time() - t0) / 3600\n    with open(res_path, "w") as f:\n        json.dump(res, f, indent=2)\n    os.remove(last_path)\n    print(f"[{run}] done: best val acc {100 * st[\'best_acc\']:.2f}% at epoch {st[\'best_epoch\']} | "\n          f"{res[\'gpu_hours\']:.2f} GPU-h" + (f" | test {100 * res[\'test_acc\']:.2f}%" if "test_acc" in res else ""))\n    return res\n\n\n# ---- results table ----------------------------------------------------------------------------\nCOLS = ["run", "pretrained_tag", "overhead_pct", "epochs_run", "best_epoch", "best_val_acc", "val_coverage",\n        "val_frac_groups_erased", "val_median_group_contrast", "test_acc", "gpu_hours", "gpu", "timm", "code_sha"]\n\n\ndef summarize(out, dst=None):\n    """Tabulate <out>/*/results.json; writes <dst>.csv and <dst>.md when dst is given."""\n    res = []\n    for p in sorted(glob.glob(os.path.join(out, "*", "results.json"))):\n        with open(p) as f:\n            res.append(json.load(f))\n    cols = [c for c in COLS if any(c in r for r in res)]  # no test column unless test was opened\n\n    def fmt(c, v):\n        if v is None:\n            return ""\n        if c.endswith("_acc"):\n            return f"{100 * v:.2f}"\n        return f"{v:.3f}" if isinstance(v, float) else str(v)\n\n    md = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]\n    md += ["| " + " | ".join(fmt(c, r.get(c)) for c in cols) + " |" for r in res]\n    md.append(f"\\nTotal GPU-hours: {sum(r[\'gpu_hours\'] for r in res):.2f} over {len(res)} runs")\n    text = "\\n".join(md)\n    if dst:\n        os.makedirs(os.path.dirname(dst) or ".", exist_ok=True)\n        with open(dst + ".csv", "w", newline="") as f:\n            w = csv.DictWriter(f, fieldnames=sorted({k for r in res for k in r}))\n            w.writeheader()\n            w.writerows(res)\n        with open(dst + ".md", "w") as f:\n            f.write(text + "\\n")\n    print(text)\n    return res\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--config", required=True)\n    ap.add_argument("--row")\n    ap.add_argument("--gate_pos", choices=["premerge", "postnorm"], help="required for row D")\n    ap.add_argument("--seed", type=int)\n    ap.add_argument("--out", help="default: output/<config name>")\n    ap.add_argument("--limit", type=int, help="smoke test: cap images per split")\n    ap.add_argument("--epochs", type=int, help="override (smoke test)")\n    ap.add_argument("--make_split", action="store_true", help="write the config\'s split file (once per dataset)")\n    ap.add_argument("--summarize", metavar="DST", help="tabulate <out>/*/results.json into DST.csv and DST.md")\n    args = ap.parse_args()\n    if args.make_split:\n        with open(args.config) as f:\n            split = make_split(json.load(f))\n        print(f"wrote {len(split[\'val\'])} val images of {split[\'n_train_total\']}")\n    elif args.summarize:\n        summarize(args.out or os.path.join("output", os.path.splitext(os.path.basename(args.config))[0]), args.summarize)\n    else:\n        train(args.config, args.row, args.gate_pos, args.seed, args.out, args.limit, args.epochs)\n\n\nif __name__ == "__main__":\n    main()\n',
    'probe_v1.py': '"""Inference-time probes on a trained v1 checkpoint (hook in model.py; see loupe_v2.probe_transform).\nAnalysis only, never selection. Use the test set: a v1 checkpoint was trained on all of train, so the\ncarved val split is not held out for it, and its \'none\' accuracy was picked on test (read the deltas).\n\n    python probe_v1.py --weights output/best_model.pth --out results/probe_v1_swin_b_test.json   # Swin-B\n    python probe_v1.py --weights ... --model swin_tiny_patch4_window7_224\n"""\nimport argparse\nimport hashlib\nimport json\nimport os\n\nimport timm\nimport torch\nimport torch.nn.functional as F\nfrom torch.utils.data import DataLoader, Subset\nfrom torchvision.datasets import ImageFolder\nfrom timm.data import create_transform\nfrom tqdm import tqdm\n\nfrom model import SwinWithLoupe\n\nMODES = ["none", "no_erasure", "no_contrast", "gate_off", "floor"]\n\n\n@torch.no_grad()\ndef run(model, loader, device, mode):\n    model.probe_mode = mode\n    correct, n, stats = 0, 0, {}\n    for images, labels in tqdm(loader, desc=mode, leave=False):\n        logits, m = model(images.to(device))\n        correct += (logits.argmax(1).cpu() == labels).sum().item()\n        n += labels.size(0)\n        if mode == "none":  # how the trained v1 mask sits relative to the LN-invariance argument\n            gmax, gmin = F.max_pool2d(m, 2), -F.max_pool2d(-m, 2)\n            for k, v in {"coverage": m.mean(), "frac_m_lt_1e-3": (m < 1e-3).float().mean(),\n                         "frac_groups_erased": (gmax < 1e-3).float().mean(),\n                         "median_group_contrast": (gmax / gmin.clamp_min(1e-6)).median()}.items():\n                stats[k] = stats.get(k, 0.0) + v.item() / len(loader)\n    return correct / n, stats\n\n\ndef probe(weights, model_name="swin_base_patch4_window7_224", data_dir="./data", split="test", modes=MODES,\n          split_file="splits/cub_val_frac0.1_seed0.json", out=None, limit=None):\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    model = SwinWithLoupe(model_name=model_name, num_classes=200, pretrained=False)\n    missing, unexpected = model.load_state_dict(torch.load(weights, map_location="cpu"), strict=False)\n    assert not missing, f"checkpoint lacks {missing[:5]}"\n    assert all(k.endswith(("relative_position_index", "attn_mask")) for k in unexpected), unexpected[:5]\n    model.to(device).eval()\n\n    tf = create_transform(input_size=224, is_training=False)\n    if split == "test":\n        ds = ImageFolder(os.path.join(data_dir, "test"), transform=tf)\n    else:\n        root = os.path.join(data_dir, "train")\n        ds = ImageFolder(root, transform=tf)\n        with open(split_file) as f:\n            keep = set(json.load(f)["val"])\n        ds = Subset(ds, [i for i, (p, _) in enumerate(ds.samples) if os.path.relpath(p, root) in keep])\n    if limit:\n        ds = Subset(ds, range(limit))\n    loader = DataLoader(ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)\n\n    with open(weights, "rb") as f:\n        sha = hashlib.sha256(f.read()).hexdigest()[:12]\n    res = {"weights": weights, "weights_sha": sha, "model": model_name, "split": split, "n_images": len(ds),\n           "timm": timm.__version__, "modes": {}}\n    for mode in modes:\n        acc, stats = run(model, loader, device, mode)\n        res["modes"][mode] = {"acc": acc, **stats}\n        print(json.dumps({"mode": mode, "acc": round(acc, 4), **{k: round(v, 4) for k, v in stats.items()}}))\n    model.probe_mode = "none"\n    if out:\n        os.makedirs(os.path.dirname(out) or ".", exist_ok=True)\n        with open(out, "w") as f:\n            json.dump(res, f, indent=2)\n    return res\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--weights", required=True)\n    ap.add_argument("--model", default="swin_base_patch4_window7_224", help="v1 default (paper model)")\n    ap.add_argument("--data_dir", default="./data")\n    ap.add_argument("--split", default="test", choices=["test", "val"])\n    ap.add_argument("--split_file", default="splits/cub_val_frac0.1_seed0.json", help="used with --split val")\n    ap.add_argument("--modes", nargs="+", default=MODES, choices=MODES)\n    ap.add_argument("--out", help="write results JSON here")\n    ap.add_argument("--limit", type=int, help="smoke test: cap images")\n    args = ap.parse_args()\n    probe(args.weights, args.model, args.data_dir, args.split, args.modes, args.split_file, args.out, args.limit)\n\n\nif __name__ == "__main__":\n    main()\n',
    'organize_files.py': '# src/organize_files.py\n\nimport pandas as pd\nimport os\nimport shutil\nfrom pathlib import Path\nfrom tqdm import tqdm\n\ndef organize_dataset(raw_data_path="CUB_200_2011", organized_data_path="data"):\n    """\n    Reads the raw CUB_200_2011 text files and organizes the images into\n    a clean train/test directory structure.\n\n    Args:\n        raw_data_path (str): The path to the downloaded, unzipped CUB_200_2011 directory.\n        organized_data_path (str): The path where the clean \'train\' and \'test\' folders will be created.\n    """\n    print("--- Starting Dataset Organization ---")\n    \n    # Ensure the raw data path exists\n    if not os.path.isdir(raw_data_path):\n        print(f"Error: Raw data directory not found at \'{raw_data_path}\'")\n        print("Please make sure you have downloaded and unzipped CUB_200_2011.tgz in your project folder.")\n        return\n\n    # Create the target directories\n    train_dir = Path(organized_data_path) / "train"\n    test_dir = Path(organized_data_path) / "test"\n    train_dir.mkdir(parents=True, exist_ok=True)\n    test_dir.mkdir(parents=True, exist_ok=True)\n    \n    # 1. Load the instruction files using pandas\n    print("Loading instruction files...")\n    images_df = pd.read_csv(Path(raw_data_path) / \'images.txt\', sep=\' \', names=[\'image_id\', \'filepath\'])\n    split_df = pd.read_csv(Path(raw_data_path) / \'train_test_split.txt\', sep=\' \', names=[\'image_id\', \'is_train\'])\n    labels_df = pd.read_csv(Path(raw_data_path) / \'image_class_labels.txt\', sep=\' \', names=[\'image_id\', \'class_id\'])\n    \n    # 2. Merge them into one master dataframe\n    master_df = images_df.merge(split_df, on=\'image_id\').merge(labels_df, on=\'image_id\')\n    print("Successfully merged metadata.")\n\n    # 3. Loop through the master dataframe and copy files\n    print(f"Organizing {len(master_df)} images into train/test folders...")\n    \n    # Using tqdm for a nice progress bar\n    for index, row in tqdm(master_df.iterrows(), total=master_df.shape[0]):\n        image_path = Path(raw_data_path) / \'images\' / row[\'filepath\']\n        \n        # The class folder name is part of the original filepath\n        # e.g., \'001.Black_footed_Albatross/...\'\n        class_folder_name = image_path.parent.name\n        \n        # Determine if it\'s a train or test image\n        if row[\'is_train\'] == 1:\n            target_dir = train_dir / class_folder_name\n        else:\n            target_dir = test_dir / class_folder_name\n            \n        # Create the class directory if it doesn\'t exist\n        target_dir.mkdir(exist_ok=True)\n        \n        # Define the destination path\n        destination_path = target_dir / image_path.name\n        \n        # Copy the file\n        shutil.copy(image_path, destination_path)\n        \n    print("--- Dataset Organization Complete! ---")\n    print(f"Clean dataset is ready at: \'{os.path.abspath(organized_data_path)}\'")\n\n\nif __name__ == "__main__":\n    # Make sure you\'ve run the wget/tar commands from a previous step\n    # to have the \'CUB_200_2011\' folder.\n    # If not, you can run them in your terminal:\n    # !wget http://www.vision.caltech.edu/visipedia-data/CUB-200-2011/CUB_200_2011.tgz\n    # !tar -xzf CUB_200_2011.tgz\n    organize_dataset()',
    'configs/screen_cub_swin_t.json': '{\n  "_notes": [\n    "Phase 1 screen (PROTOCOL.md): Swin-T on CUB-200-2011, 1 seed, L1 lambda=0.05 in every Loupe row, the",\n    "no-Loupe baseline in the same harness. Screening never opens the test set (eval_test false); set it true",\n    "only for final checkpoints. Row D\'s gate_pos is left null on purpose: run both placements unless B and",\n    "B2 differ by more than 2 points on validation, then pass --gate_pos explicitly."\n  ],\n  "timm_version": "1.0.30",\n  "model": "swin_tiny_patch4_window7_224.ms_in1k",\n  "num_classes": 200,\n  "data_dir": "./data",\n  "split_file": "splits/cub_val_frac0.1_seed0.json",\n  "val_frac": 0.1,\n  "split_seed": 0,\n  "img_size": 224,\n  "auto_augment": "rand-m9-mstd0.5",\n  "batch_size": 16,\n  "accum_steps": 2,\n  "opt": "lion",\n  "lr": 1e-5,\n  "weight_decay": 0.0,\n  "epochs": 50,\n  "eta_min": 1e-7,\n  "patience": 5,\n  "min_delta": 0.001,\n  "reg": {"kind": "l1", "lam": 0.05},\n  "seed": 0,\n  "num_workers": 2,\n  "eval_test": false,\n  "rows": {\n    "baseline": {"loupe": false},\n    "A":  {"mask_src": "s2",   "gate": "mult",     "gate_pos": "premerge"},\n    "B":  {"mask_src": "s2",   "gate": "residual", "gate_pos": "premerge", "alpha": 0.3},\n    "B2": {"mask_src": "s2",   "gate": "residual", "gate_pos": "postnorm", "alpha": 0.3},\n    "C":  {"mask_src": "s1s2", "gate": "mult",     "gate_pos": "premerge"},\n    "D":  {"mask_src": "s1s2", "gate": "residual", "gate_pos": null,       "alpha": 0.3}\n  }\n}\n',
    'splits/cub_val_frac0.1_seed0.json': '{\n"frac": 0.1,\n"seed": 0,\n"n_train_total": 5994,\n"n_classes": 200,\n"val": [\n"001.Black_footed_Albatross/Black_Footed_Albatross_0040_796066.jpg",\n"001.Black_footed_Albatross/Black_Footed_Albatross_0074_59.jpg",\n"001.Black_footed_Albatross/Black_Footed_Albatross_0080_796096.jpg",\n"002.Laysan_Albatross/Laysan_Albatross_0005_565.jpg",\n"002.Laysan_Albatross/Laysan_Albatross_0055_570.jpg",\n"002.Laysan_Albatross/Laysan_Albatross_0099_869.jpg",\n"003.Sooty_Albatross/Sooty_Albatross_0025_796361.jpg",\n"003.Sooty_Albatross/Sooty_Albatross_0043_1076.jpg",\n"003.Sooty_Albatross/Sooty_Albatross_0045_1162.jpg",\n"004.Groove_billed_Ani/Groove_Billed_Ani_0036_1604.jpg",\n"004.Groove_billed_Ani/Groove_Billed_Ani_0077_1724.jpg",\n"004.Groove_billed_Ani/Groove_Billed_Ani_0105_1562.jpg",\n"005.Crested_Auklet/Crested_Auklet_0030_794937.jpg",\n"005.Crested_Auklet/Crested_Auklet_0047_794918.jpg",\n"005.Crested_Auklet/Crested_Auklet_0073_785248.jpg",\n"006.Least_Auklet/Least_Auklet_0024_795120.jpg",\n"006.Least_Auklet/Least_Auklet_0038_795132.jpg",\n"006.Least_Auklet/Least_Auklet_0060_795095.jpg",\n"007.Parakeet_Auklet/Parakeet_Auklet_0024_2045.jpg",\n"007.Parakeet_Auklet/Parakeet_Auklet_0043_795948.jpg",\n"007.Parakeet_Auklet/Parakeet_Auklet_0080_795965.jpg",\n"008.Rhinoceros_Auklet/Rhinoceros_Auklet_0009_797539.jpg",\n"008.Rhinoceros_Auklet/Rhinoceros_Auklet_0014_797522.jpg",\n"008.Rhinoceros_Auklet/Rhinoceros_Auklet_0032_797516.jpg",\n"009.Brewer_Blackbird/Brewer_Blackbird_0026_2625.jpg",\n"009.Brewer_Blackbird/Brewer_Blackbird_0111_2613.jpg",\n"009.Brewer_Blackbird/Brewer_Blackbird_0133_2324.jpg",\n"010.Red_winged_Blackbird/Red_Winged_Blackbird_0028_4709.jpg",\n"010.Red_winged_Blackbird/Red_Winged_Blackbird_0059_4612.jpg",\n"010.Red_winged_Blackbird/Red_Winged_Blackbird_0105_4842.jpg",\n"011.Rusty_Blackbird/Rusty_Blackbird_0087_6727.jpg",\n"011.Rusty_Blackbird/Rusty_Blackbird_0101_6880.jpg",\n"011.Rusty_Blackbird/Rusty_Blackbird_0108_6867.jpg",\n"012.Yellow_headed_Blackbird/Yellow_Headed_Blackbird_0018_8588.jpg",\n"012.Yellow_headed_Blackbird/Yellow_Headed_Blackbird_0038_8689.jpg",\n"012.Yellow_headed_Blackbird/Yellow_Headed_Blackbird_0091_8555.jpg",\n"013.Bobolink/Bobolink_0008_9289.jpg",\n"013.Bobolink/Bobolink_0013_9367.jpg",\n"013.Bobolink/Bobolink_0104_10273.jpg",\n"014.Indigo_Bunting/Indigo_Bunting_0056_12637.jpg",\n"014.Indigo_Bunting/Indigo_Bunting_0082_11907.jpg",\n"014.Indigo_Bunting/Indigo_Bunting_0084_11848.jpg",\n"015.Lazuli_Bunting/Lazuli_Bunting_0039_15081.jpg",\n"015.Lazuli_Bunting/Lazuli_Bunting_0055_15043.jpg",\n"015.Lazuli_Bunting/Lazuli_Bunting_0067_14672.jpg",\n"016.Painted_Bunting/Painted_Bunting_0006_15249.jpg",\n"016.Painted_Bunting/Painted_Bunting_0055_15208.jpg",\n"016.Painted_Bunting/Painted_Bunting_0060_15224.jpg",\n"017.Cardinal/Cardinal_0045_18021.jpg",\n"017.Cardinal/Cardinal_0075_17946.jpg",\n"017.Cardinal/Cardinal_0079_19044.jpg",\n"018.Spotted_Catbird/Spotted_Catbird_0010_19436.jpg",\n"018.Spotted_Catbird/Spotted_Catbird_0027_796796.jpg",\n"018.Spotted_Catbird/Spotted_Catbird_0047_19400.jpg",\n"019.Gray_Catbird/Gray_Catbird_0090_21464.jpg",\n"019.Gray_Catbird/Gray_Catbird_0091_20416.jpg",\n"019.Gray_Catbird/Gray_Catbird_0131_19633.jpg",\n"020.Yellow_breasted_Chat/Yellow_Breasted_Chat_0012_21961.jpg",\n"020.Yellow_breasted_Chat/Yellow_Breasted_Chat_0044_22106.jpg",\n"020.Yellow_breasted_Chat/Yellow_Breasted_Chat_0075_21715.jpg",\n"021.Eastern_Towhee/Eastern_Towhee_0075_22588.jpg",\n"021.Eastern_Towhee/Eastern_Towhee_0111_22168.jpg",\n"021.Eastern_Towhee/Eastern_Towhee_0126_22639.jpg",\n"022.Chuck_will_Widow/Chuck_Will_Widow_0001_796975.jpg",\n"022.Chuck_will_Widow/Chuck_Will_Widow_0003_796977.jpg",\n"022.Chuck_will_Widow/Chuck_Will_Widow_0059_796982.jpg",\n"023.Brandt_Cormorant/Brandt_Cormorant_0032_22886.jpg",\n"023.Brandt_Cormorant/Brandt_Cormorant_0074_22881.jpg",\n"023.Brandt_Cormorant/Brandt_Cormorant_0080_23002.jpg",\n"024.Red_faced_Cormorant/Red_Faced_Cormorant_0060_23416.jpg",\n"024.Red_faced_Cormorant/Red_Faced_Cormorant_0064_23464.jpg",\n"024.Red_faced_Cormorant/Red_Faced_Cormorant_0065_796293.jpg",\n"025.Pelagic_Cormorant/Pelagic_Cormorant_0008_23602.jpg",\n"025.Pelagic_Cormorant/Pelagic_Cormorant_0062_23623.jpg",\n"025.Pelagic_Cormorant/Pelagic_Cormorant_0064_23641.jpg",\n"026.Bronzed_Cowbird/Bronzed_Cowbird_0040_24134.jpg",\n"026.Bronzed_Cowbird/Bronzed_Cowbird_0054_24159.jpg",\n"026.Bronzed_Cowbird/Bronzed_Cowbird_0086_796259.jpg",\n"027.Shiny_Cowbird/Shiny_Cowbird_0021_796870.jpg",\n"027.Shiny_Cowbird/Shiny_Cowbird_0026_796847.jpg",\n"027.Shiny_Cowbird/Shiny_Cowbird_0077_24273.jpg",\n"028.Brown_Creeper/Brown_Creeper_0058_24933.jpg",\n"028.Brown_Creeper/Brown_Creeper_0113_24560.jpg",\n"028.Brown_Creeper/Brown_Creeper_0114_24649.jpg",\n"029.American_Crow/American_Crow_0012_25305.jpg",\n"029.American_Crow/American_Crow_0036_25313.jpg",\n"029.American_Crow/American_Crow_0132_25704.jpg",\n"030.Fish_Crow/Fish_Crow_0014_26041.jpg",\n"030.Fish_Crow/Fish_Crow_0044_25964.jpg",\n"030.Fish_Crow/Fish_Crow_0083_25949.jpg",\n"031.Black_billed_Cuckoo/Black_Billed_Cuckoo_0031_26318.jpg",\n"031.Black_billed_Cuckoo/Black_Billed_Cuckoo_0054_26313.jpg",\n"031.Black_billed_Cuckoo/Black_Billed_Cuckoo_0088_26217.jpg",\n"032.Mangrove_Cuckoo/Mangrove_Cuckoo_0029_794624.jpg",\n"032.Mangrove_Cuckoo/Mangrove_Cuckoo_0034_26415.jpg",\n"032.Mangrove_Cuckoo/Mangrove_Cuckoo_0046_794616.jpg",\n"033.Yellow_billed_Cuckoo/Yellow_Billed_Cuckoo_0009_26656.jpg",\n"033.Yellow_billed_Cuckoo/Yellow_Billed_Cuckoo_0034_26694.jpg",\n"033.Yellow_billed_Cuckoo/Yellow_Billed_Cuckoo_0097_26713.jpg",\n"034.Gray_crowned_Rosy_Finch/Gray_Crowned_Rosy_Finch_0044_26976.jpg",\n"034.Gray_crowned_Rosy_Finch/Gray_Crowned_Rosy_Finch_0046_797295.jpg",\n"034.Gray_crowned_Rosy_Finch/Gray_Crowned_Rosy_Finch_0080_27108.jpg",\n"035.Purple_Finch/Purple_Finch_0011_27633.jpg",\n"035.Purple_Finch/Purple_Finch_0048_27670.jpg",\n"035.Purple_Finch/Purple_Finch_0086_27637.jpg",\n"036.Northern_Flicker/Northern_Flicker_0016_28603.jpg",\n"036.Northern_Flicker/Northern_Flicker_0054_28913.jpg",\n"036.Northern_Flicker/Northern_Flicker_0107_28667.jpg",\n"037.Acadian_Flycatcher/Acadian_Flycatcher_0019_795592.jpg",\n"037.Acadian_Flycatcher/Acadian_Flycatcher_0040_795629.jpg",\n"037.Acadian_Flycatcher/Acadian_Flycatcher_0065_29070.jpg",\n"038.Great_Crested_Flycatcher/Great_Crested_Flycatcher_0038_29533.jpg",\n"038.Great_Crested_Flycatcher/Great_Crested_Flycatcher_0070_29455.jpg",\n"038.Great_Crested_Flycatcher/Great_Crested_Flycatcher_0136_29490.jpg",\n"039.Least_Flycatcher/Least_Flycatcher_0067_30397.jpg",\n"039.Least_Flycatcher/Least_Flycatcher_0076_30117.jpg",\n"039.Least_Flycatcher/Least_Flycatcher_0093_30435.jpg",\n"040.Olive_sided_Flycatcher/Olive_Sided_Flycatcher_0020_796881.jpg",\n"040.Olive_sided_Flycatcher/Olive_Sided_Flycatcher_0035_30799.jpg",\n"040.Olive_sided_Flycatcher/Olive_Sided_Flycatcher_0052_30639.jpg",\n"041.Scissor_tailed_Flycatcher/Scissor_Tailed_Flycatcher_0005_41727.jpg",\n"041.Scissor_tailed_Flycatcher/Scissor_Tailed_Flycatcher_0092_41767.jpg",\n"041.Scissor_tailed_Flycatcher/Scissor_Tailed_Flycatcher_0111_42118.jpg",\n"042.Vermilion_Flycatcher/Vermilion_Flycatcher_0023_42565.jpg",\n"042.Vermilion_Flycatcher/Vermilion_Flycatcher_0027_42388.jpg",\n"042.Vermilion_Flycatcher/Vermilion_Flycatcher_0045_42219.jpg",\n"043.Yellow_bellied_Flycatcher/Yellow_Bellied_Flycatcher_0010_795498.jpg",\n"043.Yellow_bellied_Flycatcher/Yellow_Bellied_Flycatcher_0018_795494.jpg",\n"043.Yellow_bellied_Flycatcher/Yellow_Bellied_Flycatcher_0020_795482.jpg",\n"044.Frigatebird/Frigatebird_0012_43062.jpg",\n"044.Frigatebird/Frigatebird_0021_42913.jpg",\n"044.Frigatebird/Frigatebird_0114_42807.jpg",\n"045.Northern_Fulmar/Northern_Fulmar_0006_43753.jpg",\n"045.Northern_Fulmar/Northern_Fulmar_0070_43916.jpg",\n"045.Northern_Fulmar/Northern_Fulmar_0074_43955.jpg",\n"046.Gadwall/Gadwall_0012_30920.jpg",\n"046.Gadwall/Gadwall_0034_31212.jpg",\n"046.Gadwall/Gadwall_0060_31686.jpg",\n"047.American_Goldfinch/American_Goldfinch_0014_32154.jpg",\n"047.American_Goldfinch/American_Goldfinch_0106_32182.jpg",\n"047.American_Goldfinch/American_Goldfinch_0122_32186.jpg",\n"048.European_Goldfinch/European_Goldfinch_0012_794646.jpg",\n"048.European_Goldfinch/European_Goldfinch_0103_794664.jpg",\n"048.European_Goldfinch/European_Goldfinch_0107_794655.jpg",\n"049.Boat_tailed_Grackle/Boat_Tailed_Grackle_0016_33770.jpg",\n"049.Boat_tailed_Grackle/Boat_Tailed_Grackle_0017_33480.jpg",\n"049.Boat_tailed_Grackle/Boat_Tailed_Grackle_0113_33490.jpg",\n"050.Eared_Grebe/Eared_Grebe_0058_34174.jpg",\n"050.Eared_Grebe/Eared_Grebe_0073_34172.jpg",\n"050.Eared_Grebe/Eared_Grebe_0079_34342.jpg",\n"051.Horned_Grebe/Horned_Grebe_0072_34497.jpg",\n"051.Horned_Grebe/Horned_Grebe_0088_35023.jpg",\n"051.Horned_Grebe/Horned_Grebe_0101_35203.jpg",\n"052.Pied_billed_Grebe/Pied_Billed_Grebe_0068_35963.jpg",\n"052.Pied_billed_Grebe/Pied_Billed_Grebe_0113_35703.jpg",\n"052.Pied_billed_Grebe/Pied_Billed_Grebe_0120_35764.jpg",\n"053.Western_Grebe/Western_Grebe_0032_36439.jpg",\n"053.Western_Grebe/Western_Grebe_0057_36157.jpg",\n"053.Western_Grebe/Western_Grebe_0086_36478.jpg",\n"054.Blue_Grosbeak/Blue_Grosbeak_0031_37173.jpg",\n"054.Blue_Grosbeak/Blue_Grosbeak_0033_36980.jpg",\n"054.Blue_Grosbeak/Blue_Grosbeak_0103_36673.jpg",\n"055.Evening_Grosbeak/Evening_Grosbeak_0093_37608.jpg",\n"055.Evening_Grosbeak/Evening_Grosbeak_0102_37927.jpg",\n"055.Evening_Grosbeak/Evening_Grosbeak_0132_38025.jpg",\n"056.Pine_Grosbeak/Pine_Grosbeak_0050_38475.jpg",\n"056.Pine_Grosbeak/Pine_Grosbeak_0078_38242.jpg",\n"056.Pine_Grosbeak/Pine_Grosbeak_0107_38351.jpg",\n"057.Rose_breasted_Grosbeak/Rose_Breasted_Grosbeak_0042_39158.jpg",\n"057.Rose_breasted_Grosbeak/Rose_Breasted_Grosbeak_0068_39094.jpg",\n"057.Rose_breasted_Grosbeak/Rose_Breasted_Grosbeak_0075_39795.jpg",\n"058.Pigeon_Guillemot/Pigeon_Guillemot_0077_39885.jpg",\n"058.Pigeon_Guillemot/Pigeon_Guillemot_0081_40339.jpg",\n"058.Pigeon_Guillemot/Pigeon_Guillemot_0083_39980.jpg",\n"059.California_Gull/California_Gull_0052_41047.jpg",\n"059.California_Gull/California_Gull_0109_41465.jpg",\n"059.California_Gull/California_Gull_0135_41383.jpg",\n"060.Glaucous_winged_Gull/Glaucous_Winged_Gull_0012_44264.jpg",\n"060.Glaucous_winged_Gull/Glaucous_Winged_Gull_0051_44543.jpg",\n"060.Glaucous_winged_Gull/Glaucous_Winged_Gull_0091_44848.jpg",\n"061.Heermann_Gull/Heermann_Gull_0020_45409.jpg",\n"061.Heermann_Gull/Heermann_Gull_0086_45304.jpg",\n"061.Heermann_Gull/Heermann_Gull_0100_45440.jpg",\n"062.Herring_Gull/Herring_Gull_0051_46276.jpg",\n"062.Herring_Gull/Herring_Gull_0105_46113.jpg",\n"062.Herring_Gull/Herring_Gull_0140_46455.jpg",\n"063.Ivory_Gull/Ivory_Gull_0005_49021.jpg",\n"063.Ivory_Gull/Ivory_Gull_0037_49068.jpg",\n"063.Ivory_Gull/Ivory_Gull_0039_49412.jpg",\n"064.Ring_billed_Gull/Ring_Billed_Gull_0018_51505.jpg",\n"064.Ring_billed_Gull/Ring_Billed_Gull_0050_51567.jpg",\n"064.Ring_billed_Gull/Ring_Billed_Gull_0117_51363.jpg",\n"065.Slaty_backed_Gull/Slaty_Backed_Gull_0030_796003.jpg",\n"065.Slaty_backed_Gull/Slaty_Backed_Gull_0038_796044.jpg",\n"065.Slaty_backed_Gull/Slaty_Backed_Gull_0071_796037.jpg",\n"066.Western_Gull/Western_Gull_0025_55421.jpg",\n"066.Western_Gull/Western_Gull_0051_54320.jpg",\n"066.Western_Gull/Western_Gull_0117_44697.jpg",\n"067.Anna_Hummingbird/Anna_Hummingbird_0017_56954.jpg",\n"067.Anna_Hummingbird/Anna_Hummingbird_0047_56049.jpg",\n"067.Anna_Hummingbird/Anna_Hummingbird_0118_55773.jpg",\n"068.Ruby_throated_Hummingbird/Ruby_Throated_Hummingbird_0024_57969.jpg",\n"068.Ruby_throated_Hummingbird/Ruby_Throated_Hummingbird_0027_58191.jpg",\n"068.Ruby_throated_Hummingbird/Ruby_Throated_Hummingbird_0107_57339.jpg",\n"069.Rufous_Hummingbird/Rufous_Hummingbird_0037_60415.jpg",\n"069.Rufous_Hummingbird/Rufous_Hummingbird_0111_59408.jpg",\n"069.Rufous_Hummingbird/Rufous_Hummingbird_0120_59900.jpg",\n"070.Green_Violetear/Green_Violetear_0009_795647.jpg",\n"070.Green_Violetear/Green_Violetear_0079_795685.jpg",\n"070.Green_Violetear/Green_Violetear_0102_795702.jpg",\n"071.Long_tailed_Jaeger/Long_Tailed_Jaeger_0041_60891.jpg",\n"071.Long_tailed_Jaeger/Long_Tailed_Jaeger_0053_60906.jpg",\n"071.Long_tailed_Jaeger/Long_Tailed_Jaeger_0070_797090.jpg",\n"072.Pomarine_Jaeger/Pomarine_Jaeger_0004_795796.jpg",\n"072.Pomarine_Jaeger/Pomarine_Jaeger_0046_61301.jpg",\n"072.Pomarine_Jaeger/Pomarine_Jaeger_0055_795762.jpg",\n"073.Blue_Jay/Blue_Jay_0002_62657.jpg",\n"073.Blue_Jay/Blue_Jay_0010_61939.jpg",\n"073.Blue_Jay/Blue_Jay_0074_63487.jpg",\n"074.Florida_Jay/Florida_Jay_0029_65114.jpg",\n"074.Florida_Jay/Florida_Jay_0079_64713.jpg",\n"074.Florida_Jay/Florida_Jay_0104_64885.jpg",\n"075.Green_Jay/Green_Jay_0015_65796.jpg",\n"075.Green_Jay/Green_Jay_0044_66228.jpg",\n"075.Green_Jay/Green_Jay_0075_65701.jpg",\n"076.Dark_eyed_Junco/Dark_Eyed_Junco_0012_66932.jpg",\n"076.Dark_eyed_Junco/Dark_Eyed_Junco_0071_67426.jpg",\n"076.Dark_eyed_Junco/Dark_Eyed_Junco_0113_68470.jpg",\n"077.Tropical_Kingbird/Tropical_Kingbird_0003_69852.jpg",\n"077.Tropical_Kingbird/Tropical_Kingbird_0007_69361.jpg",\n"077.Tropical_Kingbird/Tropical_Kingbird_0081_69656.jpg",\n"078.Gray_Kingbird/Gray_Kingbird_0003_70305.jpg",\n"078.Gray_Kingbird/Gray_Kingbird_0018_70241.jpg",\n"078.Gray_Kingbird/Gray_Kingbird_0022_70155.jpg",\n"079.Belted_Kingfisher/Belted_Kingfisher_0016_70495.jpg",\n"079.Belted_Kingfisher/Belted_Kingfisher_0054_70424.jpg",\n"079.Belted_Kingfisher/Belted_Kingfisher_0080_70725.jpg",\n"080.Green_Kingfisher/Green_Kingfisher_0020_71155.jpg",\n"080.Green_Kingfisher/Green_Kingfisher_0071_71255.jpg",\n"080.Green_Kingfisher/Green_Kingfisher_0091_71248.jpg",\n"081.Pied_Kingfisher/Pied_Kingfisher_0011_72143.jpg",\n"081.Pied_Kingfisher/Pied_Kingfisher_0080_72199.jpg",\n"081.Pied_Kingfisher/Pied_Kingfisher_0128_72775.jpg",\n"082.Ringed_Kingfisher/Ringed_Kingfisher_0004_72966.jpg",\n"082.Ringed_Kingfisher/Ringed_Kingfisher_0039_71052.jpg",\n"082.Ringed_Kingfisher/Ringed_Kingfisher_0050_73002.jpg",\n"083.White_breasted_Kingfisher/White_Breasted_Kingfisher_0022_73459.jpg",\n"083.White_breasted_Kingfisher/White_Breasted_Kingfisher_0093_73311.jpg",\n"083.White_breasted_Kingfisher/White_Breasted_Kingfisher_0120_73439.jpg",\n"084.Red_legged_Kittiwake/Red_Legged_Kittiwake_0004_73811.jpg",\n"084.Red_legged_Kittiwake/Red_Legged_Kittiwake_0019_795398.jpg",\n"084.Red_legged_Kittiwake/Red_Legged_Kittiwake_0021_73808.jpg",\n"085.Horned_Lark/Horned_Lark_0013_74374.jpg",\n"085.Horned_Lark/Horned_Lark_0049_74574.jpg",\n"085.Horned_Lark/Horned_Lark_0095_74640.jpg",\n"086.Pacific_Loon/Pacific_Loon_0014_75468.jpg",\n"086.Pacific_Loon/Pacific_Loon_0023_75476.jpg",\n"086.Pacific_Loon/Pacific_Loon_0029_75495.jpg",\n"087.Mallard/Mallard_0013_77619.jpg",\n"087.Mallard/Mallard_0069_76926.jpg",\n"087.Mallard/Mallard_0077_75911.jpg",\n"088.Western_Meadowlark/Western_Meadowlark_0003_78864.jpg",\n"088.Western_Meadowlark/Western_Meadowlark_0015_78610.jpg",\n"088.Western_Meadowlark/Western_Meadowlark_0081_77798.jpg",\n"089.Hooded_Merganser/Hooded_Merganser_0023_796784.jpg",\n"089.Hooded_Merganser/Hooded_Merganser_0049_79136.jpg",\n"089.Hooded_Merganser/Hooded_Merganser_0078_78959.jpg",\n"090.Red_breasted_Merganser/Red_Breasted_Merganser_0022_79274.jpg",\n"090.Red_breasted_Merganser/Red_Breasted_Merganser_0033_79353.jpg",\n"090.Red_breasted_Merganser/Red_Breasted_Merganser_0081_79396.jpg",\n"091.Mockingbird/Mockingbird_0059_82126.jpg",\n"091.Mockingbird/Mockingbird_0094_80232.jpg",\n"091.Mockingbird/Mockingbird_0103_79659.jpg",\n"092.Nighthawk/Nighthawk_0055_83352.jpg",\n"092.Nighthawk/Nighthawk_0083_83647.jpg",\n"092.Nighthawk/Nighthawk_0090_82579.jpg",\n"093.Clark_Nutcracker/Clark_Nutcracker_0026_84945.jpg",\n"093.Clark_Nutcracker/Clark_Nutcracker_0101_85656.jpg",\n"093.Clark_Nutcracker/Clark_Nutcracker_0105_85097.jpg",\n"094.White_breasted_Nuthatch/White_Breasted_Nuthatch_0025_85803.jpg",\n"094.White_breasted_Nuthatch/White_Breasted_Nuthatch_0043_86196.jpg",\n"094.White_breasted_Nuthatch/White_Breasted_Nuthatch_0114_86554.jpg",\n"095.Baltimore_Oriole/Baltimore_Oriole_0027_87561.jpg",\n"095.Baltimore_Oriole/Baltimore_Oriole_0104_86979.jpg",\n"095.Baltimore_Oriole/Baltimore_Oriole_0120_88403.jpg",\n"096.Hooded_Oriole/Hooded_Oriole_0050_90629.jpg",\n"096.Hooded_Oriole/Hooded_Oriole_0067_90416.jpg",\n"096.Hooded_Oriole/Hooded_Oriole_0125_90382.jpg",\n"097.Orchard_Oriole/Orchard_Oriole_0009_91902.jpg",\n"097.Orchard_Oriole/Orchard_Oriole_0024_91755.jpg",\n"097.Orchard_Oriole/Orchard_Oriole_0046_91646.jpg",\n"098.Scott_Oriole/Scott_Oriole_0040_795839.jpg",\n"098.Scott_Oriole/Scott_Oriole_0056_795816.jpg",\n"098.Scott_Oriole/Scott_Oriole_0085_92206.jpg",\n"099.Ovenbird/Ovenbird_0003_92910.jpg",\n"099.Ovenbird/Ovenbird_0024_92583.jpg",\n"099.Ovenbird/Ovenbird_0104_92763.jpg",\n"100.Brown_Pelican/Brown_Pelican_0077_93464.jpg",\n"100.Brown_Pelican/Brown_Pelican_0085_95053.jpg",\n"100.Brown_Pelican/Brown_Pelican_0118_93475.jpg",\n"101.White_Pelican/White_Pelican_0053_96236.jpg",\n"101.White_Pelican/White_Pelican_0079_97380.jpg",\n"101.White_Pelican/White_Pelican_0081_96148.jpg",\n"102.Western_Wood_Pewee/Western_Wood_Pewee_0021_98101.jpg",\n"102.Western_Wood_Pewee/Western_Wood_Pewee_0041_795048.jpg",\n"102.Western_Wood_Pewee/Western_Wood_Pewee_0081_98270.jpg",\n"103.Sayornis/Sayornis_0056_99553.jpg",\n"103.Sayornis/Sayornis_0058_98798.jpg",\n"103.Sayornis/Sayornis_0099_98593.jpg",\n"104.American_Pipit/American_Pipit_0027_100189.jpg",\n"104.American_Pipit/American_Pipit_0082_99867.jpg",\n"104.American_Pipit/American_Pipit_0102_99979.jpg",\n"105.Whip_poor_Will/Whip_Poor_Will_0007_796431.jpg",\n"105.Whip_poor_Will/Whip_Poor_Will_0031_100397.jpg",\n"105.Whip_poor_Will/Whip_Poor_Will_0040_100482.jpg",\n"106.Horned_Puffin/Horned_Puffin_0001_100973.jpg",\n"106.Horned_Puffin/Horned_Puffin_0033_100731.jpg",\n"106.Horned_Puffin/Horned_Puffin_0062_100693.jpg",\n"107.Common_Raven/Common_Raven_0007_102020.jpg",\n"107.Common_Raven/Common_Raven_0054_101750.jpg",\n"107.Common_Raven/Common_Raven_0099_102534.jpg",\n"108.White_necked_Raven/White_Necked_Raven_0002_797370.jpg",\n"108.White_necked_Raven/White_Necked_Raven_0020_102730.jpg",\n"108.White_necked_Raven/White_Necked_Raven_0041_797394.jpg",\n"109.American_Redstart/American_Redstart_0027_103168.jpg",\n"109.American_Redstart/American_Redstart_0036_103231.jpg",\n"109.American_Redstart/American_Redstart_0109_103795.jpg",\n"110.Geococcyx/Geococcyx_0061_104553.jpg",\n"110.Geococcyx/Geococcyx_0078_104468.jpg",\n"110.Geococcyx/Geococcyx_0095_104358.jpg",\n"111.Loggerhead_Shrike/Loggerhead_Shrike_0041_105002.jpg",\n"111.Loggerhead_Shrike/Loggerhead_Shrike_0048_106215.jpg",\n"111.Loggerhead_Shrike/Loggerhead_Shrike_0103_105137.jpg",\n"112.Great_Grey_Shrike/Great_Grey_Shrike_0056_106752.jpg",\n"112.Great_Grey_Shrike/Great_Grey_Shrike_0058_106634.jpg",\n"112.Great_Grey_Shrike/Great_Grey_Shrike_0089_797036.jpg",\n"113.Baird_Sparrow/Baird_Sparrow_0014_794549.jpg",\n"113.Baird_Sparrow/Baird_Sparrow_0038_794592.jpg",\n"113.Baird_Sparrow/Baird_Sparrow_0048_787323.jpg",\n"114.Black_throated_Sparrow/Black_Throated_Sparrow_0027_107278.jpg",\n"114.Black_throated_Sparrow/Black_Throated_Sparrow_0035_107185.jpg",\n"114.Black_throated_Sparrow/Black_Throated_Sparrow_0086_106970.jpg",\n"115.Brewer_Sparrow/Brewer_Sparrow_0032_796702.jpg",\n"115.Brewer_Sparrow/Brewer_Sparrow_0063_796714.jpg",\n"115.Brewer_Sparrow/Brewer_Sparrow_0068_107422.jpg",\n"116.Chipping_Sparrow/Chipping_Sparrow_0001_108638.jpg",\n"116.Chipping_Sparrow/Chipping_Sparrow_0008_107810.jpg",\n"116.Chipping_Sparrow/Chipping_Sparrow_0064_108204.jpg",\n"117.Clay_colored_Sparrow/Clay_Colored_Sparrow_0029_110720.jpg",\n"117.Clay_colored_Sparrow/Clay_Colored_Sparrow_0078_110912.jpg",\n"117.Clay_colored_Sparrow/Clay_Colored_Sparrow_0087_110946.jpg",\n"118.House_Sparrow/House_Sparrow_0053_111388.jpg",\n"118.House_Sparrow/House_Sparrow_0059_111164.jpg",\n"118.House_Sparrow/House_Sparrow_0074_111997.jpg",\n"119.Field_Sparrow/Field_Sparrow_0055_113929.jpg",\n"119.Field_Sparrow/Field_Sparrow_0095_113842.jpg",\n"119.Field_Sparrow/Field_Sparrow_0106_113350.jpg",\n"120.Fox_Sparrow/Fox_Sparrow_0063_114350.jpg",\n"120.Fox_Sparrow/Fox_Sparrow_0110_115172.jpg",\n"120.Fox_Sparrow/Fox_Sparrow_0124_114662.jpg",\n"121.Grasshopper_Sparrow/Grasshopper_Sparrow_0068_115799.jpg",\n"121.Grasshopper_Sparrow/Grasshopper_Sparrow_0073_115996.jpg",\n"121.Grasshopper_Sparrow/Grasshopper_Sparrow_0119_116081.jpg",\n"122.Harris_Sparrow/Harris_Sparrow_0006_116364.jpg",\n"122.Harris_Sparrow/Harris_Sparrow_0029_116516.jpg",\n"122.Harris_Sparrow/Harris_Sparrow_0034_116439.jpg",\n"123.Henslow_Sparrow/Henslow_Sparrow_0070_796571.jpg",\n"123.Henslow_Sparrow/Henslow_Sparrow_0083_116844.jpg",\n"123.Henslow_Sparrow/Henslow_Sparrow_0106_796580.jpg",\n"124.Le_Conte_Sparrow/Le_Conte_Sparrow_0056_795178.jpg",\n"124.Le_Conte_Sparrow/Le_Conte_Sparrow_0071_795185.jpg",\n"124.Le_Conte_Sparrow/Le_Conte_Sparrow_0075_795234.jpg",\n"125.Lincoln_Sparrow/Lincoln_Sparrow_0014_117883.jpg",\n"125.Lincoln_Sparrow/Lincoln_Sparrow_0038_117461.jpg",\n"125.Lincoln_Sparrow/Lincoln_Sparrow_0088_117634.jpg",\n"126.Nelson_Sharp_tailed_Sparrow/Nelson_Sharp_Tailed_Sparrow_0013_796942.jpg",\n"126.Nelson_Sharp_tailed_Sparrow/Nelson_Sharp_Tailed_Sparrow_0022_796905.jpg",\n"126.Nelson_Sharp_tailed_Sparrow/Nelson_Sharp_Tailed_Sparrow_0045_118004.jpg",\n"127.Savannah_Sparrow/Savannah_Sparrow_0017_119171.jpg",\n"127.Savannah_Sparrow/Savannah_Sparrow_0054_120057.jpg",\n"127.Savannah_Sparrow/Savannah_Sparrow_0091_120630.jpg",\n"128.Seaside_Sparrow/Seaside_Sparrow_0039_796530.jpg",\n"128.Seaside_Sparrow/Seaside_Sparrow_0041_796529.jpg",\n"128.Seaside_Sparrow/Seaside_Sparrow_0067_796532.jpg",\n"129.Song_Sparrow/Song_Sparrow_0089_120894.jpg",\n"129.Song_Sparrow/Song_Sparrow_0091_121651.jpg",\n"129.Song_Sparrow/Song_Sparrow_0135_121261.jpg",\n"130.Tree_Sparrow/Tree_Sparrow_0007_122911.jpg",\n"130.Tree_Sparrow/Tree_Sparrow_0121_124296.jpg",\n"130.Tree_Sparrow/Tree_Sparrow_0129_124960.jpg",\n"131.Vesper_Sparrow/Vesper_Sparrow_0005_125440.jpg",\n"131.Vesper_Sparrow/Vesper_Sparrow_0022_125719.jpg",\n"131.Vesper_Sparrow/Vesper_Sparrow_0044_125470.jpg",\n"132.White_crowned_Sparrow/White_Crowned_Sparrow_0067_127576.jpg",\n"132.White_crowned_Sparrow/White_Crowned_Sparrow_0094_128627.jpg",\n"132.White_crowned_Sparrow/White_Crowned_Sparrow_0124_128082.jpg",\n"133.White_throated_Sparrow/White_Throated_Sparrow_0015_129138.jpg",\n"133.White_throated_Sparrow/White_Throated_Sparrow_0123_129009.jpg",\n"133.White_throated_Sparrow/White_Throated_Sparrow_0124_128801.jpg",\n"134.Cape_Glossy_Starling/Cape_Glossy_Starling_0043_129358.jpg",\n"134.Cape_Glossy_Starling/Cape_Glossy_Starling_0084_129346.jpg",\n"134.Cape_Glossy_Starling/Cape_Glossy_Starling_0092_129349.jpg",\n"135.Bank_Swallow/Bank_Swallow_0017_129755.jpg",\n"135.Bank_Swallow/Bank_Swallow_0049_129611.jpg",\n"135.Bank_Swallow/Bank_Swallow_0054_129743.jpg",\n"136.Barn_Swallow/Barn_Swallow_0015_132757.jpg",\n"136.Barn_Swallow/Barn_Swallow_0073_131389.jpg",\n"136.Barn_Swallow/Barn_Swallow_0077_130707.jpg",\n"137.Cliff_Swallow/Cliff_Swallow_0005_133696.jpg",\n"137.Cliff_Swallow/Cliff_Swallow_0022_133786.jpg",\n"137.Cliff_Swallow/Cliff_Swallow_0050_134054.jpg",\n"138.Tree_Swallow/Tree_Swallow_0046_135770.jpg",\n"138.Tree_Swallow/Tree_Swallow_0060_134961.jpg",\n"138.Tree_Swallow/Tree_Swallow_0102_135061.jpg",\n"139.Scarlet_Tanager/Scarlet_Tanager_0016_138008.jpg",\n"139.Scarlet_Tanager/Scarlet_Tanager_0042_138287.jpg",\n"139.Scarlet_Tanager/Scarlet_Tanager_0110_138274.jpg",\n"140.Summer_Tanager/Summer_Tanager_0001_139289.jpg",\n"140.Summer_Tanager/Summer_Tanager_0109_139522.jpg",\n"140.Summer_Tanager/Summer_Tanager_0116_139923.jpg",\n"141.Artic_Tern/Artic_Tern_0089_141652.jpg",\n"141.Artic_Tern/Artic_Tern_0099_141170.jpg",\n"141.Artic_Tern/Artic_Tern_0105_141098.jpg",\n"142.Black_Tern/Black_Tern_0013_143892.jpg",\n"142.Black_Tern/Black_Tern_0020_144163.jpg",\n"142.Black_Tern/Black_Tern_0069_144359.jpg",\n"143.Caspian_Tern/Caspian_Tern_0027_145899.jpg",\n"143.Caspian_Tern/Caspian_Tern_0096_145009.jpg",\n"143.Caspian_Tern/Caspian_Tern_0117_146009.jpg",\n"144.Common_Tern/Common_Tern_0053_148472.jpg",\n"144.Common_Tern/Common_Tern_0054_148028.jpg",\n"144.Common_Tern/Common_Tern_0094_148309.jpg",\n"145.Elegant_Tern/Elegant_Tern_0014_150523.jpg",\n"145.Elegant_Tern/Elegant_Tern_0028_151019.jpg",\n"145.Elegant_Tern/Elegant_Tern_0061_150524.jpg",\n"146.Forsters_Tern/Forsters_Tern_0066_151478.jpg",\n"146.Forsters_Tern/Forsters_Tern_0123_151789.jpg",\n"146.Forsters_Tern/Forsters_Tern_0127_150418.jpg",\n"147.Least_Tern/Least_Tern_0006_153921.jpg",\n"147.Least_Tern/Least_Tern_0016_153560.jpg",\n"147.Least_Tern/Least_Tern_0082_154396.jpg",\n"148.Green_tailed_Towhee/Green_Tailed_Towhee_0036_154875.jpg",\n"148.Green_tailed_Towhee/Green_Tailed_Towhee_0058_797399.jpg",\n"148.Green_tailed_Towhee/Green_Tailed_Towhee_0105_797438.jpg",\n"149.Brown_Thrasher/Brown_Thrasher_0010_155231.jpg",\n"149.Brown_Thrasher/Brown_Thrasher_0106_155083.jpg",\n"149.Brown_Thrasher/Brown_Thrasher_0119_155170.jpg",\n"150.Sage_Thrasher/Sage_Thrasher_0025_155661.jpg",\n"150.Sage_Thrasher/Sage_Thrasher_0040_155667.jpg",\n"150.Sage_Thrasher/Sage_Thrasher_0077_155608.jpg",\n"151.Black_capped_Vireo/Black_Capped_Vireo_0022_797459.jpg",\n"151.Black_capped_Vireo/Black_Capped_Vireo_0029_797469.jpg",\n"151.Black_capped_Vireo/Black_Capped_Vireo_0043_797458.jpg",\n"152.Blue_headed_Vireo/Blue_Headed_Vireo_0019_156311.jpg",\n"152.Blue_headed_Vireo/Blue_Headed_Vireo_0054_156455.jpg",\n"152.Blue_headed_Vireo/Blue_Headed_Vireo_0095_156092.jpg",\n"153.Philadelphia_Vireo/Philadelphia_Vireo_0012_794785.jpg",\n"153.Philadelphia_Vireo/Philadelphia_Vireo_0024_156645.jpg",\n"153.Philadelphia_Vireo/Philadelphia_Vireo_0078_794776.jpg",\n"154.Red_eyed_Vireo/Red_Eyed_Vireo_0002_156470.jpg",\n"154.Red_eyed_Vireo/Red_Eyed_Vireo_0003_157226.jpg",\n"154.Red_eyed_Vireo/Red_Eyed_Vireo_0115_157004.jpg",\n"155.Warbling_Vireo/Warbling_Vireo_0003_158484.jpg",\n"155.Warbling_Vireo/Warbling_Vireo_0017_158271.jpg",\n"155.Warbling_Vireo/Warbling_Vireo_0077_158427.jpg",\n"156.White_eyed_Vireo/White_Eyed_Vireo_0032_159145.jpg",\n"156.White_eyed_Vireo/White_Eyed_Vireo_0040_159101.jpg",\n"156.White_eyed_Vireo/White_Eyed_Vireo_0080_159087.jpg",\n"157.Yellow_throated_Vireo/Yellow_Throated_Vireo_0035_794991.jpg",\n"157.Yellow_throated_Vireo/Yellow_Throated_Vireo_0058_794994.jpg",\n"157.Yellow_throated_Vireo/Yellow_Throated_Vireo_0084_159639.jpg",\n"158.Bay_breasted_Warbler/Bay_Breasted_Warbler_0011_159736.jpg",\n"158.Bay_breasted_Warbler/Bay_Breasted_Warbler_0095_797113.jpg",\n"158.Bay_breasted_Warbler/Bay_Breasted_Warbler_0105_797143.jpg",\n"159.Black_and_white_Warbler/Black_And_White_Warbler_0078_160365.jpg",\n"159.Black_and_white_Warbler/Black_And_White_Warbler_0080_160375.jpg",\n"159.Black_and_white_Warbler/Black_And_White_Warbler_0086_160836.jpg",\n"160.Black_throated_Blue_Warbler/Black_Throated_Blue_Warbler_0060_161644.jpg",\n"160.Black_throated_Blue_Warbler/Black_Throated_Blue_Warbler_0083_161462.jpg",\n"160.Black_throated_Blue_Warbler/Black_Throated_Blue_Warbler_0106_161523.jpg",\n"161.Blue_winged_Warbler/Blue_Winged_Warbler_0036_161758.jpg",\n"161.Blue_winged_Warbler/Blue_Winged_Warbler_0054_161862.jpg",\n"161.Blue_winged_Warbler/Blue_Winged_Warbler_0091_162051.jpg",\n"162.Canada_Warbler/Canada_Warbler_0041_162418.jpg",\n"162.Canada_Warbler/Canada_Warbler_0047_162439.jpg",\n"162.Canada_Warbler/Canada_Warbler_0113_162403.jpg",\n"163.Cape_May_Warbler/Cape_May_Warbler_0022_162912.jpg",\n"163.Cape_May_Warbler/Cape_May_Warbler_0023_163133.jpg",\n"163.Cape_May_Warbler/Cape_May_Warbler_0069_162980.jpg",\n"164.Cerulean_Warbler/Cerulean_Warbler_0038_797230.jpg",\n"164.Cerulean_Warbler/Cerulean_Warbler_0047_797211.jpg",\n"164.Cerulean_Warbler/Cerulean_Warbler_0076_163294.jpg",\n"165.Chestnut_sided_Warbler/Chestnut_Sided_Warbler_0006_163858.jpg",\n"165.Chestnut_sided_Warbler/Chestnut_Sided_Warbler_0018_164148.jpg",\n"165.Chestnut_sided_Warbler/Chestnut_Sided_Warbler_0043_164114.jpg",\n"166.Golden_winged_Warbler/Golden_Winged_Warbler_0021_794821.jpg",\n"166.Golden_winged_Warbler/Golden_Winged_Warbler_0024_794807.jpg",\n"166.Golden_winged_Warbler/Golden_Winged_Warbler_0071_164370.jpg",\n"167.Hooded_Warbler/Hooded_Warbler_0008_164641.jpg",\n"167.Hooded_Warbler/Hooded_Warbler_0040_165173.jpg",\n"167.Hooded_Warbler/Hooded_Warbler_0104_164982.jpg",\n"168.Kentucky_Warbler/Kentucky_Warbler_0008_165369.jpg",\n"168.Kentucky_Warbler/Kentucky_Warbler_0055_165426.jpg",\n"168.Kentucky_Warbler/Kentucky_Warbler_0069_165318.jpg",\n"169.Magnolia_Warbler/Magnolia_Warbler_0004_165535.jpg",\n"169.Magnolia_Warbler/Magnolia_Warbler_0023_165827.jpg",\n"169.Magnolia_Warbler/Magnolia_Warbler_0032_165960.jpg",\n"170.Mourning_Warbler/Mourning_Warbler_0012_166515.jpg",\n"170.Mourning_Warbler/Mourning_Warbler_0026_166538.jpg",\n"170.Mourning_Warbler/Mourning_Warbler_0052_166537.jpg",\n"171.Myrtle_Warbler/Myrtle_Warbler_0050_166820.jpg",\n"171.Myrtle_Warbler/Myrtle_Warbler_0059_166946.jpg",\n"171.Myrtle_Warbler/Myrtle_Warbler_0072_166702.jpg",\n"172.Nashville_Warbler/Nashville_Warbler_0032_167385.jpg",\n"172.Nashville_Warbler/Nashville_Warbler_0035_167283.jpg",\n"172.Nashville_Warbler/Nashville_Warbler_0051_167250.jpg",\n"173.Orange_crowned_Warbler/Orange_Crowned_Warbler_0049_167974.jpg",\n"173.Orange_crowned_Warbler/Orange_Crowned_Warbler_0066_167949.jpg",\n"173.Orange_crowned_Warbler/Orange_Crowned_Warbler_0091_167578.jpg",\n"174.Palm_Warbler/Palm_Warbler_0003_170474.jpg",\n"174.Palm_Warbler/Palm_Warbler_0028_169670.jpg",\n"174.Palm_Warbler/Palm_Warbler_0136_170276.jpg",\n"175.Pine_Warbler/Pine_Warbler_0017_171678.jpg",\n"175.Pine_Warbler/Pine_Warbler_0074_172061.jpg",\n"175.Pine_Warbler/Pine_Warbler_0126_171282.jpg",\n"176.Prairie_Warbler/Prairie_Warbler_0020_173359.jpg",\n"176.Prairie_Warbler/Prairie_Warbler_0051_172585.jpg",\n"176.Prairie_Warbler/Prairie_Warbler_0112_173383.jpg",\n"177.Prothonotary_Warbler/Prothonotary_Warbler_0063_173830.jpg",\n"177.Prothonotary_Warbler/Prothonotary_Warbler_0066_174318.jpg",\n"177.Prothonotary_Warbler/Prothonotary_Warbler_0110_173857.jpg",\n"178.Swainson_Warbler/Swainson_Warbler_0011_174680.jpg",\n"178.Swainson_Warbler/Swainson_Warbler_0022_794868.jpg",\n"178.Swainson_Warbler/Swainson_Warbler_0030_794864.jpg",\n"179.Tennessee_Warbler/Tennessee_Warbler_0021_174761.jpg",\n"179.Tennessee_Warbler/Tennessee_Warbler_0052_175089.jpg",\n"179.Tennessee_Warbler/Tennessee_Warbler_0100_175168.jpg",\n"180.Wilson_Warbler/Wilson_Warbler_0054_175285.jpg",\n"180.Wilson_Warbler/Wilson_Warbler_0073_175876.jpg",\n"180.Wilson_Warbler/Wilson_Warbler_0126_175368.jpg",\n"181.Worm_eating_Warbler/Worm_Eating_Warbler_0012_795515.jpg",\n"181.Worm_eating_Warbler/Worm_Eating_Warbler_0034_795560.jpg",\n"181.Worm_eating_Warbler/Worm_Eating_Warbler_0092_795524.jpg",\n"182.Yellow_Warbler/Yellow_Warbler_0018_176674.jpg",\n"182.Yellow_Warbler/Yellow_Warbler_0056_176320.jpg",\n"182.Yellow_Warbler/Yellow_Warbler_0095_176202.jpg",\n"183.Northern_Waterthrush/Northern_Waterthrush_0009_177078.jpg",\n"183.Northern_Waterthrush/Northern_Waterthrush_0049_177173.jpg",\n"183.Northern_Waterthrush/Northern_Waterthrush_0051_177120.jpg",\n"184.Louisiana_Waterthrush/Louisiana_Waterthrush_0025_177403.jpg",\n"184.Louisiana_Waterthrush/Louisiana_Waterthrush_0026_177503.jpg",\n"184.Louisiana_Waterthrush/Louisiana_Waterthrush_0062_177285.jpg",\n"185.Bohemian_Waxwing/Bohemian_Waxwing_0013_796660.jpg",\n"185.Bohemian_Waxwing/Bohemian_Waxwing_0092_796666.jpg",\n"185.Bohemian_Waxwing/Bohemian_Waxwing_0117_177858.jpg",\n"186.Cedar_Waxwing/Cedar_Waxwing_0016_178629.jpg",\n"186.Cedar_Waxwing/Cedar_Waxwing_0093_178139.jpg",\n"186.Cedar_Waxwing/Cedar_Waxwing_0130_178308.jpg",\n"187.American_Three_toed_Woodpecker/American_Three_Toed_Woodpecker_0016_179927.jpg",\n"187.American_Three_toed_Woodpecker/American_Three_Toed_Woodpecker_0029_796143.jpg",\n"187.American_Three_toed_Woodpecker/American_Three_Toed_Woodpecker_0041_796150.jpg",\n"188.Pileated_Woodpecker/Pileated_Woodpecker_0011_180519.jpg",\n"188.Pileated_Woodpecker/Pileated_Woodpecker_0087_179959.jpg",\n"188.Pileated_Woodpecker/Pileated_Woodpecker_0112_180439.jpg",\n"189.Red_bellied_Woodpecker/Red_Bellied_Woodpecker_0010_180772.jpg",\n"189.Red_bellied_Woodpecker/Red_Bellied_Woodpecker_0021_182303.jpg",\n"189.Red_bellied_Woodpecker/Red_Bellied_Woodpecker_0104_181616.jpg",\n"190.Red_cockaded_Woodpecker/Red_Cockaded_Woodpecker_0004_182340.jpg",\n"190.Red_cockaded_Woodpecker/Red_Cockaded_Woodpecker_0018_794705.jpg",\n"190.Red_cockaded_Woodpecker/Red_Cockaded_Woodpecker_0027_794713.jpg",\n"191.Red_headed_Woodpecker/Red_Headed_Woodpecker_0005_183414.jpg",\n"191.Red_headed_Woodpecker/Red_Headed_Woodpecker_0012_183395.jpg",\n"191.Red_headed_Woodpecker/Red_Headed_Woodpecker_0052_182702.jpg",\n"192.Downy_Woodpecker/Downy_Woodpecker_0038_184418.jpg",\n"192.Downy_Woodpecker/Downy_Woodpecker_0062_184343.jpg",\n"192.Downy_Woodpecker/Downy_Woodpecker_0066_183953.jpg",\n"193.Bewick_Wren/Bewick_Wren_0065_185247.jpg",\n"193.Bewick_Wren/Bewick_Wren_0067_184816.jpg",\n"193.Bewick_Wren/Bewick_Wren_0135_185251.jpg",\n"194.Cactus_Wren/Cactus_Wren_0078_185899.jpg",\n"194.Cactus_Wren/Cactus_Wren_0097_186015.jpg",\n"194.Cactus_Wren/Cactus_Wren_0108_185733.jpg",\n"195.Carolina_Wren/Carolina_Wren_0035_186356.jpg",\n"195.Carolina_Wren/Carolina_Wren_0055_186154.jpg",\n"195.Carolina_Wren/Carolina_Wren_0099_186237.jpg",\n"196.House_Wren/House_Wren_0035_187708.jpg",\n"196.House_Wren/House_Wren_0071_187399.jpg",\n"196.House_Wren/House_Wren_0087_187946.jpg",\n"197.Marsh_Wren/Marsh_Wren_0033_188778.jpg",\n"197.Marsh_Wren/Marsh_Wren_0075_188549.jpg",\n"197.Marsh_Wren/Marsh_Wren_0107_188390.jpg",\n"198.Rock_Wren/Rock_Wren_0019_188968.jpg",\n"198.Rock_Wren/Rock_Wren_0021_189175.jpg",\n"198.Rock_Wren/Rock_Wren_0045_189153.jpg",\n"199.Winter_Wren/Winter_Wren_0015_190099.jpg",\n"199.Winter_Wren/Winter_Wren_0026_189816.jpg",\n"199.Winter_Wren/Winter_Wren_0066_189637.jpg",\n"200.Common_Yellowthroat/Common_Yellowthroat_0093_190609.jpg",\n"200.Common_Yellowthroat/Common_Yellowthroat_0106_190989.jpg",\n"200.Common_Yellowthroat/Common_Yellowthroat_0121_190597.jpg"\n]\n}',
}

for path, text in FILES.items():
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w") as f:
        f.write(text)
import importlib, loupe_v2, model, train_v2, probe_v1
for m in (loupe_v2, model, train_v2, probe_v1):
    importlib.reload(m)
print("wrote", ", ".join(FILES))

In [ ]:
#@title Get CUB-200-2011 (downloaded once, then cached on Drive)
import hashlib, shutil
CUB_URL = "https://data.caltech.edu/records/65de6-vp158/files/CUB_200_2011.tgz?download=1"
CUB_MD5 = "97eceeb196236b17998738112f37df78"
TGZ, CACHE = "/content/CUB_200_2011.tgz", f"{DRIVE_DIR}/cache/CUB_200_2011.tgz"

def md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 24), b""):
            h.update(chunk)
    return h.hexdigest()

def n_images(d):
    return sum(len(files) for _, _, files in os.walk(d))

if not (os.path.isdir("data/train") and n_images("data/train") == 5994 and n_images("data/test") == 5794):
    if os.path.exists(CACHE):
        shutil.copy(CACHE, TGZ)
    else:
        !curl -L --fail -o {TGZ} "{CUB_URL}"
    assert md5(TGZ) == CUB_MD5, f"CUB archive checksum mismatch; delete {CACHE} if it exists and rerun"
    if not os.path.exists(CACHE):
        shutil.copy(TGZ, CACHE)
    !tar -xzf {TGZ} -C /content
    from organize_files import organize_dataset
    shutil.rmtree("data", ignore_errors=True)
    organize_dataset("/content/CUB_200_2011", "data")
print("train", n_images("data/train"), "| test", n_images("data/test"))

## Phase 1
**1a. v1 probe.** The paper's v1 checkpoint is run on the test set under five inference-time modes (`none`, `no_erasure`, `no_contrast`, `gate_off`, `floor`) and its mask statistics are recorded (`frac_groups_erased` and others).
This is analysis, not selection. The v1 checkpoint was trained on all of `train` and picked on test, so read the differences between modes rather than the absolute accuracy.

**1b–1c. Screen.** The baseline and rows A, B, B2 and C run first. Row D then runs in both placements, unless B and B2 differ by more than 2 points on validation (about 600 images, so smaller gaps are noise).

**Decision table.** Validation only. It is saved to `MyDrive/Loupe/results/` as CSV and markdown, together with GPU-hours. Stop there and pick the core v2 configuration before Phase 2.

In [ ]:
#@title 1a. Probe the paper's v1 Swin-B checkpoint (five modes plus mask statistics, test set)
if not PROBE_V1:
    print("PROBE_V1 is off")
elif not os.path.exists(V1_CKPT):
    print(f"Skipped: no checkpoint at {V1_CKPT}")
else:
    name = os.path.splitext(os.path.basename(V1_CKPT))[0]
    out = f"{DRIVE_DIR}/results/probe_{name}_{'smoke' if SMOKE_TEST else 'test'}.json"
    if os.path.exists(out) and not SMOKE_TEST:
        print(open(out).read())
    else:
        probe_v1.probe(V1_CKPT, V1_MODEL, "data", "test", out=out, limit=64 if SMOKE_TEST else None)

In [ ]:
#@title 1b. Baseline and rows A, B, B2, C (finished runs are skipped, interrupted runs resume)
CFG = "configs/screen_cub_swin_t.json"
OUT = f"{DRIVE_DIR}/output/" + ("smoke" if SMOKE_TEST else "screen_cub_swin_t")
KW = dict(limit=64, epochs=1) if SMOKE_TEST else {}
for row in [r.strip() for r in ROWS.split(",") if r.strip()]:
    train_v2.train(CFG, row, seed=SEED, out=OUT, **KW)

In [ ]:
#@title 1c. Row D: both placements, unless B and B2 differ by more than 2 points on validation
if RUN_D:
    acc = {r: train_v2.load_result(OUT, r, SEED)["best_val_acc"] for r in ("B", "B2")}
    gap = 100 * (acc["B"] - acc["B2"])
    places = ["premerge", "postnorm"] if abs(gap) <= 2 else ["premerge" if gap > 0 else "postnorm"]
    print(f"val acc: B (premerge) {100 * acc['B']:.2f} | B2 (postnorm) {100 * acc['B2']:.2f} | "
          f"gap {gap:+.2f} pts -> D in {places}")
    for gp in places:
        train_v2.train(CFG, "D", gate_pos=gp, seed=SEED, out=OUT, **KW)

In [ ]:
#@title Decision table (validation only), saved to MyDrive/Loupe/results/
_ = train_v2.summarize(OUT, f"{DRIVE_DIR}/results/{os.path.basename(OUT)}")